## Chain rule

The chain rule tells you how to differentiate a **composition** of functions.

If `y = f(g(x))`, set `u = g(x)`. Then:

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$$

In words: **multiply the derivatives along the chain.** The rate at which `y` changes
with `x` is the rate `y` changes with `u`, times the rate `u` changes with `x`.

For a longer chain `y = f(g(h(x)))` you just keep multiplying:

$$\frac{dy}{dx} = \frac{dy}{du}\cdot\frac{du}{dv}\cdot\frac{dv}{dx}$$

**Why it matters for us:** backprop *is* the chain rule. A neural net is one big
composition of functions; to get the gradient of the loss w.r.t. any weight, you
multiply local derivatives backward from the loss to that weight.

**Useful resources**

- Parr & Howard (2018) — **The Matrix Calculus You Need For Deep Learning**
  <https://arxiv.org/abs/1802.01528> — §2-4 build the chain rule up to the vector/matrix form
  backprop actually uses. The chain rule itself is Leibniz (1676), not a paper you can link.
- 3Blue1Brown — *Essence of Calculus*, ch. 4 (visual intuition for why the derivatives multiply).

In [1]:
import numpy as np

# Example: y = sin(x**2).  Let u = x**2, so y = sin(u).
#   dy/du = cos(u) = cos(x**2)
#   du/dx = 2x
#   dy/dx = cos(x**2) * 2x     <-- chain rule

def y(x):
    return np.sin(x**2)

def dy_dx_analytic(x):
    return np.cos(x**2) * 2*x

# Numerical check via central finite difference: (f(x+h) - f(x-h)) / 2h
def dy_dx_numeric(x, h=1e-5):
    return (y(x + h) - y(x - h)) / (2*h)

x = 1.3
print('analytic:', dy_dx_analytic(x))
print('numeric :', dy_dx_numeric(x))

analytic: -0.3091960801711924
numeric : -0.3091960803947025


## Partial derivatives

For a function of **several variables**, there's no single slope — it depends on which
direction you move. So we take the derivative **one variable at a time**, holding the
others fixed. The curly `∂` (vs. `d`) is the reminder that "everything else is frozen":

$$\frac{\partial f}{\partial x} = \text{rate } f \text{ changes as } x \text{ moves, with all other inputs held constant}$$

**Example:** `f(x, z) = x**2 * z + sin(z)`

- `∂f/∂x`: treat `z` as constant → `2xz`  (the `sin(z)` term is constant in `x`, so → 0)
- `∂f/∂z`: treat `x` as constant → `x**2 + cos(z)`

**The gradient** stacks all partials into a vector, pointing in the direction of
steepest increase (gradient descent steps along its negative):

$$\nabla f = \left[\frac{\partial f}{\partial x},\ \frac{\partial f}{\partial z}\right]$$

**Why it matters for us:** the loss depends on millions of weights. The gradient is the
vector of partials w.r.t. every weight — each computed by the chain rule while holding
the others fixed. That whole vector is what backprop produces.

**Useful resources**

- Parr & Howard (2018) — **The Matrix Calculus You Need For Deep Learning**
  <https://arxiv.org/abs/1802.01528> — §3 is partial derivatives → gradients → Jacobians, in
  exactly the notation deep learning uses.

In [2]:
# f(x, z) = x**2 * z + sin(z)
def f(x, z):
    return x**2 * z + np.sin(z)

# Analytic partials
def grad_analytic(x, z):
    df_dx = 2*x*z
    df_dz = x**2 + np.cos(z)
    return np.array([df_dx, df_dz])

# Numeric gradient: central difference, perturbing ONE variable at a time.
def grad_numeric(x, z, h=1e-5):
    df_dx = (f(x + h, z) - f(x - h, z)) / (2*h)   # z held fixed
    df_dz = (f(x, z + h) - f(x, z - h)) / (2*h)   # x held fixed
    return np.array([df_dx, df_dz])

x, z = 1.3, 0.7
print('analytic grad:', grad_analytic(x, z))
print('numeric  grad:', grad_numeric(x, z))

analytic grad: [1.82       2.45484219]
numeric  grad: [1.82       2.45484219]


## Gradients of scalar functions

A **scalar function** maps a vector to a single number: `f: ℝⁿ → ℝ`. Think of the loss:
many inputs (all the weights), one output (the loss value).

Its **gradient** is the vector holding the partial derivative w.r.t. *each* input:

$$\nabla f(\mathbf{x}) = \left[\frac{\partial f}{\partial x_1},\ \frac{\partial f}{\partial x_2},\ \dots,\ \frac{\partial f}{\partial x_n}\right]$$

Key facts:

- **Same shape as the input.** If `x` is a length-`n` vector, `∇f` is also length `n`.
  (If `x` is a matrix of weights, `∇f` is a matrix of the same shape.)
- **Direction of steepest ascent.** `∇f` points where `f` increases fastest; `−∇f` is the
  descent direction. Gradient descent is literally `x ← x − lr · ∇f`.
- **Zero at flat points.** At a minimum/maximum/saddle, `∇f = 0`.

**The numeric gradient checker (generalized).** The per-variable loop from before becomes:
perturb each component `x_i` by `±h`, hold the rest fixed, take a central difference. This
gives a slow but formula-free gradient — the tool we'll use to verify every backward pass
in this project.

**Useful resources**

- Parr & Howard (2018) — **The Matrix Calculus You Need For Deep Learning**
  <https://arxiv.org/abs/1802.01528> — the reference for gradient shapes ("the gradient has the
  same shape as the thing you differentiate by").

In [3]:
# A scalar function of a VECTOR:  f(x) = sum_i (x_i**2)  +  x_0 * x_1
#   -> maps R^n to a single number.
def f(x):
    return np.sum(x**2) + x[0] * x[1]

# Analytic gradient (worked out by hand, component by component):
#   d/dx_i of sum(x^2) = 2 x_i
#   the extra x0*x1 term adds x1 to component 0, and x0 to component 1
def grad_analytic(x):
    g = 2 * x.copy()
    g[0] += x[1]
    g[1] += x[0]
    return g

# General numeric gradient checker: loops over EVERY component,
# perturbs just that one by +/- h, central difference. Works for any f: R^n -> R.
def numeric_gradient(f, x, h=1e-5):
    grad = np.zeros_like(x, dtype=float)
    for i in range(x.size):
        step = np.zeros_like(x, dtype=float)
        step[i] = h
        grad[i] = (f(x + step) - f(x - step)) / (2*h)
    return grad

x = np.array([1.0, 2.0, -3.0, 0.5])
print('analytic:', grad_analytic(x))
print('numeric :', numeric_gradient(f, x))
print('max abs diff:', np.max(np.abs(grad_analytic(x) - numeric_gradient(f, x))))

analytic: [ 4.  5. -6.  1.]
numeric : [ 4.  5. -6.  1.]
max abs diff: 1.2812417793384157e-10


## Softmax

**Softmax** turns a vector of arbitrary real numbers (**logits**) into a **probability
distribution** — all entries in `(0, 1)` and summing to `1`:

$$\text{softmax}(x)_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

How to read it:

- **Exponentiate** every logit → all values become positive.
- **Normalize** by the sum → they become probabilities that add to 1.
- It's **monotonic**: the largest logit gets the largest probability. It's a *soft* argmax —
  instead of picking one winner, it spreads weight, mostly onto the top entries.

Properties that matter later:

- **Shift-invariant:** adding a constant to every logit doesn't change the output (this is
  exactly what the log-sum-exp trick exploits below).
- **Not scale-invariant:** multiplying logits by a factor sharpens (large factor → near
  one-hot) or flattens (small factor → near uniform) the distribution. That factor is the
  **temperature** knob used in sampling.

**Why it matters for us:** it's the output layer of a classifier / language model — it
converts the network's raw scores into `p(class)` or `p(next token)`, and it pairs with
cross-entropy loss (Step 2).

**Useful resources**

- **Original paper:** Bridle (1990) — *Probabilistic Interpretation of Feedforward Classification
  Network Outputs, with Relationships to Statistical Pattern Recognition* — where the softmax
  output layer was introduced and named. (In *Neurocomputing*, NATO ASI Series vol. 68.)
- The function itself is older: it is the **Boltzmann/Gibbs distribution** from statistical
  physics, with the logits as negative energies and the temperature as `T`.

In [4]:
# Softmax: logits -> probability distribution
def softmax(x):
    e = np.exp(x - np.max(x))   # stable form (see next section for why)
    return e / np.sum(e)

logits = np.array([2.0, 1.0, 0.1])
p = softmax(logits)
print('probs      :', p)
print('sum to 1   :', p.sum())
print('argmax kept:', np.argmax(logits) == np.argmax(p))   # largest logit -> largest prob

# Temperature: scaling logits sharpens or flattens the distribution
for T in [0.5, 1.0, 5.0]:
    print(f'T={T}: {np.round(softmax(logits / T), 3)}')   # small T -> peaky, large T -> flat

probs      : [0.65900114 0.24243297 0.09856589]
sum to 1   : 1.0
argmax kept: True
T=0.5: [0.864 0.117 0.019]
T=1.0: [0.659 0.242 0.099]
T=5.0: [0.4   0.327 0.273]


## Numerical stability: floating point, overflow, log-sum-exp

### Floating point
Computers store reals in finite bits (float64 = 64 bits). Two consequences:

- **Finite range.** The largest float64 is about `1.8e308`. Anything bigger becomes `inf`
  (**overflow**); anything tiny underflows to `0.0`.
- **Finite precision.** Only ~15–16 significant decimal digits. `0.1 + 0.2 != 0.3` exactly,
  and subtracting two nearly-equal numbers throws away digits (**catastrophic cancellation**).

### Where it bites us: softmax
Softmax needs exponentials: `softmax(x)_i = exp(x_i) / Σ_j exp(x_j)`. But `exp` grows
insanely fast — `exp(1000)` is `inf` in float64. Then `inf / inf = nan` and the whole
forward pass is poisoned. Logits in a real net easily reach these ranges.

### The log-sum-exp trick
Softmax is **shift-invariant**: subtracting the same constant `c` from every logit leaves
the result unchanged, because the constant cancels top and bottom:

$$\frac{e^{x_i - c}}{\sum_j e^{x_j - c}} = \frac{e^{-c}\,e^{x_i}}{e^{-c}\sum_j e^{x_j}} = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

Choose `c = max(x)`. Now the largest exponent is `exp(0) = 1` — no overflow — and every
other term is between 0 and 1. Same math, safe numbers.

The same idea gives a stable **log-sum-exp**, `log Σ exp(x_j) = c + log Σ exp(x_j − c)`,
which is how cross-entropy loss is computed directly from logits (never forming the raw
`exp`).

**Useful resources**

- Goldberg (1991) — **What Every Computer Scientist Should Know About Floating-Point Arithmetic**
  <https://docs.oracle.com/cd/E19957-01/806-3568/ncg_goldberg.html> — the standard reference for
  why finite precision behaves the way it does (rounding, cancellation, overflow).

In [5]:
# Floating-point limits
print('largest float64 :', np.finfo(np.float64).max)
print('exp(1000)       :', np.exp(1000.0))      # -> inf (overflow)
print('0.1 + 0.2 == 0.3:', 0.1 + 0.2 == 0.3)    # -> False (finite precision)

# Naive softmax: overflows on large logits
def softmax_naive(x):
    e = np.exp(x)
    return e / np.sum(e)

# Stable softmax: subtract the max first (log-sum-exp trick)
def softmax_stable(x):
    e = np.exp(x - np.max(x))
    return e / np.sum(e)

x = np.array([1000.0, 1001.0, 1002.0])   # big logits
print('\nnaive :', softmax_naive(x))     # -> [nan nan nan]
print('stable:', softmax_stable(x))      # -> correct, finite

# Same answer as naive on SMALL inputs (shift-invariance, sanity check)
small = np.array([1.0, 2.0, 3.0])
print('\nagree on small inputs:', np.allclose(softmax_naive(small), softmax_stable(small)))

# Stable log-sum-exp: log(sum(exp(x))) without ever forming exp(x) directly
def logsumexp(x):
    c = np.max(x)
    return c + np.log(np.sum(np.exp(x - c)))

print('logsumexp(big logits):', logsumexp(x))   # finite, ~1002.4

largest float64 : 1.7976931348623157e+308
exp(1000)       : inf
0.1 + 0.2 == 0.3: False

naive : [nan nan nan]
stable: [0.09003057 0.24472847 0.66524096]

agree on small inputs: True
logsumexp(big logits): 1002.4076059644444


/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/ipykernel_launcher.py:3: RuntimeWarning: overflow encountered in exp
  This is separate from the ipykernel package so we can avoid doing imports until
/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/ipykernel_launcher.py:8: RuntimeWarning: overflow encountered in exp
  
/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/ipykernel_launcher.py:9: RuntimeWarning: invalid value encountered in true_divide
  if __name__ == "__main__":


## Backprop

Training = minimize a scalar loss `L` over millions of parameters via gradient descent,
which needs `∂L/∂θ` for every parameter `θ`. **Backprop** computes all of them exactly in
**one forward + one backward pass** (vs. `numeric_gradient`, which needs 2 passes *per
parameter*).

It's the **chain rule applied backward** through the network:

- **Forward:** run input → output, caching intermediate values.
- **Backward:** start at the loss (`∂L/∂L = 1`) and walk backward. Each layer receives the
  upstream gradient `∂L/∂(its output)` and returns `∂L/∂(params)` (to update) and
  `∂L/∂(its input)` (passed to the previous layer).

Efficient because the loss is a single scalar, so one backward sweep reaches every
parameter (reverse-mode autodiff). `numeric_gradient` stays as the checker that verifies
each hand-derived backward is correct.

**Useful resources**

- **Original paper:** Rumelhart, Hinton & Williams (1986) — **Learning representations by
  back-propagating errors**, *Nature* 323, 533-536 <https://doi.org/10.1038/323533a0> — the paper
  that made backprop the way neural networks are trained.
- Linnainmaa (1970) — the earlier origin of reverse-mode automatic differentiation itself
  (master's thesis, Univ. of Helsinki), which backprop is a special case of.
- Karpathy — **micrograd** <https://github.com/karpathy/micrograd> — backprop as ~100 lines of
  readable Python; the fastest way to see the mechanism.

## Activations: sigmoid, tanh, ReLU, GELU

Without a nonlinearity between linear layers, the whole stack collapses to one linear map.
Activations add the curvature that lets a net approximate non-linear functions.

| Name | Formula | Range | Derivative |
|---|---|---|---|
| **sigmoid** | `1 / (1 + e^-x)` | (0, 1) | `s(x)·(1 − s(x))` |
| **tanh** | `(e^x − e^-x)/(e^x + e^-x)` | (−1, 1) | `1 − tanh(x)²` |
| **ReLU** | `max(0, x)` | [0, ∞) | `1 if x>0 else 0` |
| **GELU** | `x · Φ(x)` (Φ = normal CDF) | ≈[−0.17, ∞) | smooth, ≈ReLU |

- **sigmoid / tanh** squash into a fixed range but **saturate** (flat tails → tiny
  gradients). tanh is zero-centered, usually preferred over sigmoid for hidden layers.
- **ReLU** is cheap and doesn't saturate for `x>0` → the default for deep nets.
- **GELU** is a smooth ReLU used in Transformers (what we'll use).

**Useful resources**

- **ReLU:** Nair & Hinton (2010) — **Rectified Linear Units Improve Restricted Boltzmann
  Machines** <https://icml.cc/Conferences/2010/papers/432.pdf> — where ReLU entered deep learning.
- **GELU:** Hendrycks & Gimpel (2016) — **Gaussian Error Linear Units (GELUs)**
  <https://arxiv.org/abs/1606.08415> — the activation the Transformer we're building uses.
- sigmoid/tanh predate deep learning (logistic function: Verhulst, 1838); no origin paper to link.

In [6]:
import numpy as np

def sigmoid(x):   return 1 / (1 + np.exp(-x))
def d_sigmoid(x): s = sigmoid(x); return s * (1 - s)

def tanh(x):      return np.tanh(x)
def d_tanh(x):    return 1 - np.tanh(x) ** 2

def relu(x):      return np.maximum(0, x)
def d_relu(x):    return (x > 0).astype(float)

# GELU (tanh approximation) and its derivative
def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

# Check a derivative numerically (central difference) at a point
x0 = 0.7
num = (sigmoid(x0 + 1e-5) - sigmoid(x0 - 1e-5)) / (2e-5)
print('d_sigmoid analytic:', d_sigmoid(x0))
print('d_sigmoid numeric :', num)

d_sigmoid analytic: 0.22171287329310904
d_sigmoid numeric : 0.22171287329397768


## Weight initialization (why scale matters: Xavier/He)

Weights start random (to break symmetry) but their **scale** is critical. Each layer
multiplies its input by `W`; if `W` is too large the signal (and gradients) **explode**
across layers, too small and they **vanish**.

The fix: scale the initial variance by the layer width so signal magnitude is preserved
layer to layer:

- **Xavier/Glorot** — `Var(W) = 1 / n_in` (or `2/(n_in+n_out)`). For tanh/sigmoid.
- **He** — `Var(W) = 2 / n_in`. For ReLU (accounts for ReLU zeroing half the inputs).

Practically: `W = randn(n_in, n_out) * sqrt(scale / n_in)`.

**Useful resources**

- **Xavier/Glorot init — original paper:** Glorot & Bengio (2010) — **Understanding the difficulty
  of training deep feedforward neural networks**
  <https://proceedings.mlr.press/v9/glorot10a.html> — derives the `1/n_in` variance rule.
- **He init — original paper:** He et al. (2015) — **Delving Deep into Rectifiers**
  <https://arxiv.org/abs/1502.01852> — the `2/n_in` variant for ReLU networks.

## Vanishing / exploding gradients

Backprop multiplies many local derivatives along the path from loss to an early layer. If
those factors are consistently **< 1**, the product shrinks toward 0 (**vanishing** — early
layers barely learn); if **> 1**, it blows up (**exploding** — unstable, `nan`).

This is why deep nets were historically hard to train. Mitigations we'll use:
- **ReLU/GELU** (derivative 1 for active units, doesn't saturate)
- **good init** (Xavier/He keeps factors near 1)
- **residual connections** and **LayerNorm** (Step 5) — keep gradients flowing through depth.

**Useful resources**

- **Original analysis:** Hochreiter (1991) — *Untersuchungen zu dynamischen neuronalen Netzen*
  (diploma thesis, TU Munich) — the first identification of the vanishing-gradient problem.
- Bengio, Simard & Frasconi (1994) — *Learning long-term dependencies with gradient descent is
  difficult*, IEEE Trans. Neural Networks 5(2) — the formal result in English.
- Pascanu, Mikolov & Bengio (2013) — **On the difficulty of training Recurrent Neural Networks**
  <https://arxiv.org/abs/1211.5063> — the modern treatment, and the source of **gradient
  clipping**.

## Universal approximation theorem

A feedforward net with a **single hidden layer** and a nonlinear activation can approximate
*any* continuous function on a bounded region, to arbitrary accuracy — given enough hidden
units.

Caveats worth knowing:
- It's an **existence** result: it says such weights exist, not that gradient descent will
  find them.
- "Enough units" can be impractically large. **Depth** (many layers) expresses many
  functions far more efficiently than one very wide layer — which is why we go deep.

**Useful resources**

- **Original paper:** Cybenko (1989) — **Approximation by superpositions of a sigmoidal function**,
  *Math. Control Signals Systems* 2, 303-314 <https://doi.org/10.1007/BF02551274> — the first proof,
  for sigmoidal activations.
- Hornik (1991) — *Approximation capabilities of multilayer feedforward networks*, Neural Networks
  4(2) — generalizes it: the power comes from the architecture, not the specific activation.

## Dead ReLUs & activation saturation

Two failure modes where neurons stop learning because their **gradient is ~0**:

- **Dead ReLU** — if a ReLU's input is always negative, its output is always 0 and its
  derivative is always 0, so no gradient flows and the weights never update — the neuron is
  permanently "dead." Caused by bad init or too-large learning rates. Mitigated by careful
  init, lower lr, or variants (LeakyReLU, GELU).
- **Saturation** — sigmoid/tanh have flat tails; for large `|x|` the derivative → 0, so
  gradients vanish there too. Another reason ReLU-family activations are preferred in the
  hidden layers of deep nets.

**Useful resources**

- Maas, Hannun & Ng (2013) — *Rectifier Nonlinearities Improve Neural Network Acoustic Models*
  (ICML workshop) — introduces **Leaky ReLU** as the fix for dead units.
- Nair & Hinton (2010) — **Rectified Linear Units Improve Restricted Boltzmann Machines**
  <https://icml.cc/Conferences/2010/papers/432.pdf> — the original ReLU paper, for context.

## Cross-entropy loss

The loss for classification. The model outputs probabilities `p = softmax(logits)` over
classes; the true label is a class index (equivalently a **one-hot** vector). Cross-entropy
measures how far `p` is from the truth:

$$L = -\sum_c \text{onehot}_c \, \log p_c = -\log p_{\text{correct}}$$

Since one-hot is 0 everywhere except the true class, it collapses to **just the negative log
of the probability assigned to the correct class**.

### Intuition — it rewards confident-correct, punishes confident-wrong

The `-log` shape is the whole story:

| p(correct) | `-log p` = loss | meaning |
|---|---|---|
| 1.0 | 0.00 | perfect, no loss |
| 0.9 | 0.11 | confident & right → tiny loss |
| 0.5 | 0.69 | unsure → moderate loss |
| 0.1 | 2.30 | confident & **wrong** → big loss |
| 0.01 | 4.61 | very confident & wrong → huge loss |
| → 0 | → ∞ | assigning ~0 to the truth is catastrophic |

So it isn't enough to rank the right class highest — the model is pushed to put **high
probability** on it. Being confidently wrong is punished far more than being unsure.

### Worked example (3 classes, true class = 0)

- `p = [0.9, 0.05, 0.05]` → `L = -log(0.9) = 0.105`  (good)
- `p = [0.5, 0.3, 0.2]`  → `L = -log(0.5) = 0.693`  (unsure)
- `p = [0.1, 0.6, 0.3]`  → `L = -log(0.1) = 2.303`  (confident, wrong)

Only `p[0]` (the true class) enters the loss — the other entries matter only because
softmax makes them compete for the same probability mass.

### Why `log`

Log turns "probability of getting everything right" (a product) into a **sum** of per-example
terms — nicer to optimize — and its explosion near 0 (`-log(x) → ∞`) is what makes the model
*terrified* of assigning near-zero probability to the truth.

### Batched form

Average the per-example loss over the batch:  `L = mean over examples of -log p_correct`.

### The clean combined gradient

Softmax and cross-entropy are derived *together*, because the messy pieces cancel into one
strikingly simple result:

$$\frac{\partial L}{\partial \text{logits}} = p - \text{onehot} = \text{softmax} - \text{onehot}$$

Read it as **"predicted distribution minus true distribution"**. Example (true class 0,
`p = [0.9, 0.05, 0.05]`):

$$p - \text{onehot} = [0.9-1,\; 0.05,\; 0.05] = [-0.1,\; +0.05,\; +0.05]$$

Negative on the correct class (gradient descent will **raise** that logit), positive on the
wrong ones (**lower** them). This is why the output layer's backward is a one-liner — you
never differentiate softmax and log separately. Compute it straight from **logits** (with the
stable log-sum-exp) for numerical safety.

**Useful resources**

- **Original paper (softmax + cross-entropy together):** Bridle (1990) — *Probabilistic
  Interpretation of Feedforward Classification Network Outputs…* — introduces the softmax output
  layer trained by maximum likelihood, which **is** cross-entropy.
- The measure itself comes from information theory: Shannon (1948) — *A Mathematical Theory of
  Communication*, *Bell System Technical Journal* 27 — cross-entropy = expected bits to encode the
  true distribution using the model's.

In [7]:
import numpy as np

def softmax(x):
    e = np.exp(x - np.max(x)); return e / e.sum()

def cross_entropy(logits, target):      # target = correct class index
    return -np.log(softmax(logits)[target])

# --- how loss depends on confidence in the CORRECT class (true = 0) ---
print('loss vs. probability on the correct class:')
for p_correct in [1.0, 0.9, 0.5, 0.1, 0.01]:
    print(f'  p(correct)={p_correct:>4}  ->  loss = {-np.log(p_correct):.3f}')

# --- three predictions, same true class 0 ---
print('\nthree predictions (true class = 0):')
for name, probs in [('good', [0.9,0.05,0.05]),
                    ('unsure', [0.5,0.3,0.2]),
                    ('confident-wrong', [0.1,0.6,0.3])]:
    print(f'  {name:16s} p={probs}  loss={-np.log(probs[0]):.3f}')

# --- the combined gradient  softmax - onehot ---
logits = np.array([2.0, 0.5, -1.0, 1.0]); target = 2
onehot = np.zeros_like(logits); onehot[target] = 1.0
analytic = softmax(logits) - onehot
numeric = np.array([
    (cross_entropy(logits + e, target) - cross_entropy(logits - e, target)) / 2e-5
    for e in (np.eye(len(logits)) * 1e-5)
])
print('\ngradient (softmax - onehot):')
print('  analytic:', analytic.round(4))
print('  numeric :', numeric.round(4), '  <- matches')

loss vs. probability on the correct class:
  p(correct)= 1.0  ->  loss = -0.000
  p(correct)= 0.9  ->  loss = 0.105
  p(correct)= 0.5  ->  loss = 0.693
  p(correct)= 0.1  ->  loss = 2.303
  p(correct)=0.01  ->  loss = 4.605

three predictions (true class = 0):
  good             p=[0.9, 0.05, 0.05]  loss=0.105
  unsure           p=[0.5, 0.3, 0.2]  loss=0.693
  confident-wrong  p=[0.1, 0.6, 0.3]  loss=2.303

gradient (softmax - onehot):
  analytic: [ 0.6095  0.136  -0.9697  0.2242]
  numeric : [ 0.6095  0.136  -0.9697  0.2242]   <- matches


## Gradient descent: SGD, learning rate, mini-batches

Weights move downhill against the gradient:

$$\theta \leftarrow \theta - \eta \, \nabla_\theta L$$

- **Learning rate `η`** — the step size. Too small → painfully slow; too large → overshoots,
  loss oscillates or diverges. The single most important hyperparameter.
- **Mini-batches** — estimate the gradient from a small chunk of data per step (instead of the
  whole dataset), giving many more updates per pass. Some noise, which is usually fine.
- **SGD** (stochastic gradient descent) — the name for gradient descent using mini-batch
  (or single-example) gradient estimates. The baseline optimizer everything else builds on.

**Useful resources**

- **Original paper (SGD):** Robbins & Monro (1951) — **A Stochastic Approximation Method**,
  *Annals of Mathematical Statistics* 22(3)
  <https://projecteuclid.org/journals/annals-of-mathematical-statistics/volume-22/issue-3/A-Stochastic-Approximation-Method/10.1214/aoms/1177729586.full>
  — where updating from noisy gradient estimates was introduced and proved to converge.
- Plain gradient descent is older still: Cauchy (1847).

## Adam

Plain SGD uses **one fixed learning rate for every parameter**. That struggles when different
parameters need different step sizes, or when gradients are noisy. **Adam** fixes this by
tracking two running averages of the gradient, **per parameter**.

### The two moments

- **`m` — 1st moment (momentum):** an exponential moving average of recent gradients.
  Smooths the direction and carries velocity through flat or noisy regions (like a ball
  rolling downhill instead of reacting to every bump).
- **`v` — 2nd moment (variance):** an exponential moving average of recent *squared*
  gradients. Used to scale the step **per parameter**: a parameter with consistently large
  gradients gets a *smaller* effective step, a rarely-updated one gets a *larger* step.

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\,g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)\,g_t^2$$

$$\hat m_t = \frac{m_t}{1-\beta_1^{\,t}}, \quad \hat v_t = \frac{v_t}{1-\beta_2^{\,t}}, \qquad
\theta_t = \theta_{t-1} - \eta \, \frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}$$

### Why bias correction

`m` and `v` start at **0**, so the first few averages are biased toward 0 (too small). Dividing
by `(1 - \beta^t)` undoes this. Concretely, at step `t=1` with `g=1`, `\beta_1=0.9`:

- raw `m = 0.9·0 + 0.1·1 = 0.1`  → without correction the first step is ~10× too small
- corrected `\hat m = 0.1 / (1 - 0.9^1) = 0.1 / 0.1 = 1.0`  → back to the true gradient

As `t` grows, `\beta^t → 0` and the correction fades away (it only matters early).

### The effective step

`\hat m / (\sqrt{\hat v} + \epsilon)` is roughly **"direction, normalized by typical
magnitude"** — so the step size is close to `\eta` regardless of how big or small the raw
gradients are. That self-scaling is why Adam trains fast with little tuning.

### Defaults

`\beta_1 = 0.9`, `\beta_2 = 0.999`, `\epsilon = 1e-8`, `\eta ≈ 1e-3`. Robust out of the box —
which is why Adam (and AdamW) is the standard optimizer for Transformers.

**Useful resources**

- **Original paper:** Kingma & Ba (2014) — **Adam: A Method for Stochastic Optimization**
  <https://arxiv.org/abs/1412.6980> — §2 is the algorithm box, §3 the bias-correction derivation.

In [8]:
import numpy as np

# Minimize f(x) = x^2  (min at x=0, gradient = 2x).  Compare SGD vs Adam.
def sgd(lr=0.1, steps=40):
    x = 5.0
    for _ in range(steps):
        g = 2*x
        x -= lr*g
    return x

def adam(lr=0.1, steps=40, b1=0.9, b2=0.999, eps=1e-8):
    x = 5.0; m = 0.0; v = 0.0
    for t in range(1, steps+1):
        g = 2*x
        m = b1*m + (1-b1)*g          # 1st moment
        v = b2*v + (1-b2)*g*g        # 2nd moment
        mhat = m / (1 - b1**t)       # bias correction
        vhat = v / (1 - b2**t)
        x -= lr * mhat / (np.sqrt(vhat) + eps)
        if t in (1, 2, 5, 40):
            print(f'  step {t:2d}: x={x:+.4f}  m={m:+.3f}  v={v:.3f}  mhat={mhat:+.3f}')
    return x

print('Adam trajectory minimizing x^2 from x=5:')
xa = adam()
print(f'\nfinal x  -> SGD: {sgd():.4f}   Adam: {xa:.4f}   (target 0.0)')

Adam trajectory minimizing x^2 from x=5:
  step  1: x=+4.9000  m=+1.000  v=0.100  mhat=+10.000
  step  2: x=+4.8001  m=+1.880  v=0.196  mhat=+9.895
  step  5: x=+4.5010  m=+3.914  v=0.460  mhat=+9.558
  step 40: x=+1.4757  m=+4.324  v=1.728  mhat=+4.389

final x  -> SGD: 0.0007   Adam: 1.4757   (target 0.0)


## Optimizers landscape (good to know)

- **SGD + momentum** — add a velocity term to SGD; cheap, strong generalization (common in
  vision).
- **RMSProp** — per-parameter adaptive step from squared-gradient average (Adam's `v` half).
- **Adam** — momentum + RMSProp combined; the general-purpose default.
- **AdamW** — Adam with **decoupled weight decay** (L2 applied directly to weights, not folded
  into the gradient). The standard for training Transformers today.

**Useful resources**

- **AdamW — original paper:** Loshchilov & Hutter (2017) — **Decoupled Weight Decay
  Regularization** <https://arxiv.org/abs/1711.05101> — the optimizer used to train Transformers.
- **RMSProp:** Tieleman & Hinton (2012), Coursera *Neural Networks for Machine Learning*, lecture
  6e — famously never published as a paper; cite the lecture.
- **Momentum:** Polyak (1964) — *Some methods of speeding up the convergence of iteration methods*.

## Learning-rate schedules (good to know)

The learning rate usually shouldn't stay constant:

- **Warmup** — start `η` tiny and ramp up over the first steps; avoids early instability while
  Adam's moment estimates are still noisy.
- **Decay** — shrink `η` over training (cosine or linear) so steps get finer as you near a
  minimum. **Warmup-then-cosine-decay** is the typical Transformer recipe.

**Useful resources**

- **Cosine decay + restarts — original paper:** Loshchilov & Hutter (2016) — **SGDR: Stochastic
  Gradient Descent with Warm Restarts** <https://arxiv.org/abs/1608.03983>.
- **Warmup** was introduced in the Transformer paper itself: Vaswani et al. (2017), §5.3
  <https://arxiv.org/abs/1706.03762>.

## Regularization (good to know)

Techniques that fight **overfitting** (memorizing train data instead of generalizing):

- **L2 / weight decay** — penalize large weights → simpler, smoother models.
- **Dropout** — randomly zero some activations during training → prevents co-dependence.
- **Early stopping** — stop when *validation* loss stops improving.
- **Label smoothing** — soften one-hot targets (e.g. 0.9/0.1) → less overconfident, better
  calibrated.

**Useful resources**

- **Dropout — original paper:** Srivastava et al. (2014) — **Dropout: A Simple Way to Prevent
  Neural Networks from Overfitting** <https://jmlr.org/papers/v15/srivastava14a.html>.
- **Label smoothing — original paper:** Szegedy et al. (2015) — **Rethinking the Inception
  Architecture for Computer Vision** <https://arxiv.org/abs/1512.00567>, §7.
- **Weight decay / L2** predates deep learning; the modern decoupled form is AdamW
  <https://arxiv.org/abs/1711.05101>.

## Metrics: accuracy vs. loss vs. perplexity (good to know)

- **Loss** — what you optimize (cross-entropy); smooth and differentiable, but not intuitive.
- **Accuracy** — fraction of correct predictions; intuitive, but not differentiable (can't
  train on it directly) and coarse.
- **Perplexity** — the language-model metric, `exp(cross-entropy)`. Read it as "on average the
  model is this unsure among how many choices" — perplexity 1 = perfect, `vocab_size` = random
  guessing. We'll use it in Step 8.

**Useful resources**

- **Perplexity — original use:** Jelinek, Mercer, Bahl & Baker (1977) — *Perplexity — a measure of
  the difficulty of speech recognition tasks*, JASA 62(S1) — where the metric was named.
- Any modern LM paper reports it; the Transformer and GPT papers are the natural comparisons.

# Step 3 — Tokenizer, Embeddings & Language Modeling

Everything below is the language machinery our captioning model reuses: how text becomes
numbers, how numbers become learnable vectors, and what "predict the next token" means.

## Language modeling: next-token prediction

A **language model** assigns probability to a sequence of tokens, and — the useful part —
predicts the **next token** given everything before it:

$$p(w_t \mid w_1, w_2, \dots, w_{t-1})$$

By the **chain rule of probability**, the probability of a whole sequence factorizes into a
product of next-token probabilities:

$$p(w_1, \dots, w_T) = \prod_{t=1}^{T} p(w_t \mid w_{<t})$$

This is **autoregressive**: generate left to right, each step conditioning on the tokens
already produced. Training maximizes the probability of the real next token at every
position — equivalently, **minimizes cross-entropy** (Step 2) between the predicted
distribution and the actual next token (a one-hot).

- **Objective per position:** "predict next token" is a classification over the vocabulary.
  Loss = cross-entropy(logits_t, w_t), averaged over all positions.
- **Perplexity** = exp(average cross-entropy): roughly "how many choices the model is
  effectively unsure among" (see the Step 2 metrics cell).

**Why it matters for us:** our captioning model *is* this — next-token prediction over caption
tokens, conditioned on image tokens. Steps 4–5 add attention so the conditioning becomes
powerful.

**Useful resources**

- **Original paper (neural LM):** Bengio, Ducharme, Vincent & Jauvin (2003) — **A Neural
  Probabilistic Language Model** <https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf> —
  introduces next-token prediction with learned embeddings, the direct ancestor of what we build.
- The statistical framing is older: Shannon (1948) — *A Mathematical Theory of Communication*.

In [9]:
import numpy as np

# Toy 3-token vocab {0:'a', 1:'b', 2:'.'}. A model gives next-token distributions
# (rows sum to 1) that depend on the current token:  p(next | current)
P = {
    0: np.array([0.1, 0.7, 0.2]),   # after 'a'
    1: np.array([0.6, 0.1, 0.3]),   # after 'b'
    2: np.array([0.5, 0.4, 0.1]),   # after '.'
}
itos = {0: 'a', 1: 'b', 2: '.'}

seq = [0, 1, 2]                      # the sequence "a b ."
logp = 0.0
for current, nxt in zip(seq[:-1], seq[1:]):
    p = P[current][nxt]
    logp += np.log(p)
    print(f"p({itos[nxt]} | {itos[current]}) = {p}")

n = len(seq) - 1
print("sequence probability :", np.exp(logp))       # product of the conditionals
print("avg cross-entropy    :", -logp / n)          # mean negative log-likelihood
print("perplexity           :", np.exp(-logp / n))  # exp(cross-entropy)

p(b | a) = 0.7
p(. | b) = 0.3
sequence probability : 0.20999999999999996
avg cross-entropy    : 0.7803238741323343
perplexity           : 2.182178902359924


## Tokenization

Models work with **integers**, not raw text. A **tokenizer** converts between text and a
sequence of integer **token ids**, using a fixed **vocabulary** (the set of known tokens).

- **encode:** `"hi"` -> `[7, 12]`
- **decode:** `[7, 12]` -> `"hi"`

**Granularity** — what counts as one token:

| Level | Token = | Vocab size | Sequence length | Notes |
|---|---|---|---|---|
| **character** | one character | tiny (~100) | long | trivial to build; no `<unk>` if all chars seen |
| **word** | one word | large (10k-100k+) | short | needs `<unk>` for unseen words |
| **sub-word (BPE)** | frequent chunks | medium (~30k-50k) | medium | best of both; what real LLMs use |

**Special tokens** — reserved ids with structural meaning:

- `<bos>` — beginning of sequence (the "start generating" signal)
- `<eos>` — end of sequence (tells generation to stop)
- `<pad>` — filler so batched sequences share a length (masked out of the loss)
- `<unk>` — unknown token, for inputs outside the vocabulary

**Why it matters for us:** we build a **character-level** tokenizer (simplest, no `<unk>`), add
`<bos>`/`<eos>` so the model knows where a caption starts and stops, and later `<pad>` to batch
captions of different lengths.

**Useful resources**

- Character-level tokenization has no origin paper — it is the trivial baseline. The papers worth
  reading are the sub-word ones in the next section (BPE, WordPiece, SentencePiece).
- Karpathy — **Let's build the GPT Tokenizer** <https://www.youtube.com/watch?v=zduSFxRajkE> —
  builds BPE from scratch and explains why tokenization causes so many LLM quirks.

In [10]:
import numpy as np

class CharTokenizer:
    # Character-level tokenizer: text <-> list of integer ids.

    def __init__(self, text: str) -> None:
        special_tokens: list = ["<pad>", "<bos>", "<eos>"]
        unique_characters: list = sorted(set(text))
        self.id_to_token: list = special_tokens + unique_characters
        self.token_to_id: dict = {token: i for i, token in enumerate(self.id_to_token)}

    @property
    def vocab_size(self) -> int:
        return len(self.id_to_token)

    def encode(self, text: str, add_specials: bool = True) -> list:
        token_ids: list = [self.token_to_id[character] for character in text]
        if add_specials:
            token_ids = [self.token_to_id["<bos>"]] + token_ids + [self.token_to_id["<eos>"]]
        return token_ids

    def decode(self, token_ids: list, skip_specials: bool = True) -> str:
        specials: set = {"<pad>", "<bos>", "<eos>"}
        return "".join(
            self.id_to_token[i] for i in token_ids
            if not (skip_specials and self.id_to_token[i] in specials)
        )

tokenizer = CharTokenizer("a dog runs")
print("vocab size :", tokenizer.vocab_size)
encoded = tokenizer.encode("a dog")
print("encode     :", encoded)
print("decode     :", repr(tokenizer.decode(encoded)))
print("round-trip :", tokenizer.decode(tokenizer.encode("a dog")) == "a dog")

vocab size : 12
encode     : [1, 4, 3, 5, 8, 6, 2]
decode     : 'a dog'
round-trip : True


## Sub-word tokenization: BPE, WordPiece, SentencePiece (good to know)

Character-level makes long sequences; word-level makes huge vocabularies and chokes on unseen
words. **Sub-word** tokenizers split text into frequent chunks — common words stay whole, rare
words break into pieces:

- **BPE (Byte-Pair Encoding)** — start from characters, repeatedly merge the most frequent
  adjacent pair into a new token until the vocab hits a target size. GPT uses byte-level BPE
  (`tiktoken`).
- **WordPiece** — like BPE but merges by likelihood gain rather than raw frequency (BERT).
- **SentencePiece** — trains directly on raw text (no pre-tokenization); language-agnostic.

Benefits: fixed vocab, no `<unk>` (anything decomposes to sub-words/bytes), shorter sequences
than char-level. The assignment permits an existing tokenizer (e.g. `tiktoken`) — but
char-level is enough for our small corpus and keeps everything from scratch.

**Useful resources**

- **BPE — original paper:** Sennrich, Haddow & Birch (2015) — **Neural Machine Translation of Rare
  Words with Subword Units** <https://arxiv.org/abs/1508.07909> — brings byte-pair encoding
  (Gage, 1994, a compression algorithm) into NLP.
- **WordPiece:** Schuster & Nakajima (2012) — *Japanese and Korean Voice Search*, ICASSP.
- **SentencePiece:** Kudo & Richardson (2018) <https://arxiv.org/abs/1808.06226>.

## Embeddings

A token id is just an integer label — no meaning, no notion of "close to." An **embedding**
maps each id to a **learned dense vector**, so the model can represent similarity and
structure:

$$\text{id } i \;\longrightarrow\; E[i] \in \mathbb{R}^{d}$$

The **embedding table** `E` has shape `(vocab_size, d)` — one row per token. "Embedding a
token" is just **selecting its row**: `E[i]`.

### It equals one-hot @ E, but cheaper
Selecting row `i` equals multiplying a one-hot vector (1 at position `i`) by `E`:

$$\text{onehot}(i)\, E = E[i]$$

So a lookup is a matmul with a one-hot input — but we skip building the one-hot and just
**index**. Same result, far cheaper.

- The table `E` is a **trainable parameter**; its rows are learned by backprop like any weight.
- After training, similar tokens end up with nearby vectors (see Word2Vec below).

**Why it matters for us:** we have a **token embedding** for caption tokens, and the **patch
embedding** projects image patches into the *same* `d`-dim space so attention can mix them.
Positions get embeddings too (below).

**Useful resources**

- **Original paper:** Bengio et al. (2003) — **A Neural Probabilistic Language Model**
  <https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf> — introduces learned distributed
  word vectors trained jointly with the model, exactly our embedding table.
- Mikolov et al. (2013) — **Efficient Estimation of Word Representations in Vector Space**
  <https://arxiv.org/abs/1301.3781> — Word2Vec; made embeddings a standalone artifact.

In [11]:
import numpy as np

vocab_size, d = 6, 4
rng = np.random.default_rng(0)
embedding_table = rng.standard_normal((vocab_size, d))   # one row per token

token_ids = np.array([2, 0, 4, 2])          # a sequence of ids
embedded = embedding_table[token_ids]        # lookup -> shape (len, d)
print("embedded shape:", embedded.shape)

# Equivalence: one-hot @ E selects the same rows
one_hot = np.zeros((len(token_ids), vocab_size))
one_hot[np.arange(len(token_ids)), token_ids] = 1
print("lookup == onehot @ E:", np.allclose(embedded, one_hot @ embedding_table))

embedded shape: (4, 4)
lookup == onehot @ E: True


## Embedding backward: scatter-add

Forward is a lookup: `embedded[t] = E[ids[t]]`. So the gradient w.r.t. the table `E` only
touches the **rows that were used**, and each used row receives the upstream gradient of that
position:

$$\frac{\partial L}{\partial E[i]} = \sum_{t \,:\, \text{ids}[t]=i} \text{d_embedded}[t]$$

The **sum** is the key: if a token id appears **multiple times**, its row gets a contribution
from *each* occurrence — they **add up**. This is a **scatter-add**: scatter each position's
gradient into the row of its id, accumulating on collisions.

- Naive `d_table[ids] = d_embedded` is **wrong** — it overwrites on duplicate ids instead of summing.
- Correct: `np.add.at(d_table, ids, d_embedded)` (or an explicit loop with `+=`).

**Why it matters for us:** captions repeat characters/words constantly (spaces, "a", "the"),
so this accumulation is the norm, not an edge case.

**Useful resources**

- No origin paper — scatter-add is the mechanical consequence of differentiating a lookup, not an
  idea someone published. The embedding layer it belongs to is Bengio et al. (2003)
  <https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf>.
- NumPy docs — `np.add.at` <https://numpy.org/doc/stable/reference/generated/numpy.ufunc.at.html> —
  the unbuffered in-place operation that makes duplicate indices accumulate instead of overwrite.
- PyTorch calls the same operation `index_add_` / `scatter_add_`; worth reading to confirm the
  frameworks do exactly what we do by hand.

In [12]:
import numpy as np

vocab_size, d = 5, 3
token_ids = np.array([1, 3, 1, 1])                                  # id 1 appears 3x
d_embedded = np.arange(1, 4 * d + 1, dtype=float).reshape(4, d)     # upstream grad per position

# WRONG: assignment overwrites duplicates (id 1 keeps only its last gradient)
d_table_wrong = np.zeros((vocab_size, d))
d_table_wrong[token_ids] = d_embedded

# RIGHT: scatter-add accumulates duplicates
d_table_right = np.zeros((vocab_size, d))
np.add.at(d_table_right, token_ids, d_embedded)

print("row for id=1, wrong :", d_table_wrong[1])
print("row for id=1, right :", d_table_right[1], "(sum of its 3 occurrences)")
print("manual sum          :", d_embedded[[0, 2, 3]].sum(axis=0))

row for id=1, wrong : [10. 11. 12.]
row for id=1, right : [18. 21. 24.] (sum of its 3 occurrences)
manual sum          : [18. 21. 24.]


## Word2Vec & embedding geometry (good to know)

Trained embeddings arrange meaning **geometrically**:

- **Similar words are close** (by cosine similarity) — "cat" near "dog".
- **Directions encode relations** — the famous `king - man + woman ~= queen`: analogies become
  vector arithmetic.

**Word2Vec** (2013) learned such vectors from context (predict a word from its neighbors, or
vice versa). Our embeddings are learned end-to-end for the task rather than separately, but the
same geometry emerges — which is why the grounding analysis (Step 8) can ask whether a caption
word's representation lines up with the right image region.

**Useful resources**

- **Original paper:** Mikolov, Chen, Corrado & Dean (2013) — **Efficient Estimation of Word
  Representations in Vector Space** <https://arxiv.org/abs/1301.3781> — CBOW and skip-gram.
- Mikolov et al. (2013b) — *Distributed Representations of Words and Phrases and their
  Compositionality* <https://arxiv.org/abs/1310.4546> — negative sampling, and the
  `king − man + woman ≈ queen` analogy results.

## N-gram models and the bigram LM

Before neural nets, language models were **n-grams**: estimate `p(w_t | previous n-1 tokens)`
by **counting** occurrences in a corpus.

- **Markov assumption:** only the last `n-1` tokens matter (a fixed, short context).
- **bigram (n=2):** `p(w_t | w_{t-1})` — condition on just the previous token. The simplest
  possible LM.

**Counting bigrams:** build `counts[i, j]` = "how often token `j` follows token `i`", then
normalize each row to probabilities. No learning — pure statistics.

**Neural bigram (our version):** replace the count table with a tiny network — embed the
current token, project to vocab logits, softmax — trained with cross-entropy. It *learns* the
same conditional distribution, and unlike counts it **scales** to real context via attention
(Steps 4-5). The bigram is our bridge from "counting" to "a trained LM."

**Limits of n-grams:** context capped at `n-1`; the table grows as `vocab^n` (sparse,
memory-hungry); unseen n-grams need smoothing. Embeddings + attention lift all three limits —
the reason the field moved to neural models.

**Useful resources**

- **Original paper:** Shannon (1948) — *A Mathematical Theory of Communication*, *Bell System
  Technical Journal* 27, 379-423 — §3 builds n-gram approximations of English and generates text
  from them; the first language model.
- Chen & Goodman (1996/1999) — *An Empirical Study of Smoothing Techniques for Language Modeling* —
  the reference on the smoothing that n-grams need for unseen contexts.

In [13]:
import numpy as np

text = "abracadabra"
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
V = len(chars)

# Count bigrams: counts[i, j] = # times j follows i
counts = np.zeros((V, V))
for a, b in zip(text[:-1], text[1:]):
    counts[stoi[a], stoi[b]] += 1

# Row-normalize to p(next | current)
probs = counts / counts.sum(axis=1, keepdims=True)
print("chars        :", chars)
print("p(next | 'a'):", np.round(probs[stoi['a']], 2))

# Generate by sampling from the bigram distribution
rng = np.random.default_rng(0)
out = ['a']
for _ in range(10):
    current = stoi[out[-1]]
    nxt = rng.choice(V, p=probs[current])
    out.append(itos[nxt])
print("sample       :", "".join(out))

chars        : ['a', 'b', 'c', 'd', 'r']
p(next | 'a'): [0.   0.5  0.25 0.25 0.  ]
sample       : acabradacad


## Teacher forcing vs. autoregressive generation

The same model is used two ways; the input differs.

**Training — teacher forcing.** Feed the **real** sequence; the target at each position is the
**next real token**. Inputs and targets are the same sequence shifted by one:

```
tokens : <bos>  a    dog   runs
inputs : <bos>  a    dog   runs
targets:  a     dog  runs  <eos>
```

Every position trains in parallel (with a causal mask, Step 4) on the ground-truth prefix —
fast and stable, because the model always sees correct history.

**Generation — autoregressive.** No target exists; feed the model's **own** previous outputs:
predict a token, append it, feed it back, repeat until `<eos>`. Slower (one token at a time)
and errors can compound.

**Exposure bias** — the mismatch: at train time the model only sees *correct* prefixes, but at
generation it must handle its *own* (possibly wrong) prefixes. A known limitation; at our scale
we accept it.

**Why it matters for us:** training feeds ground-truth captions (teacher forcing); the demo
generates captions autoregressively from an image.

**Useful resources**

- **Original paper:** Williams & Zipser (1989) — **A Learning Algorithm for Continually Running
  Fully Recurrent Neural Networks**, *Neural Computation* 1(2), 270-280
  <https://doi.org/10.1162/neco.1989.1.2.270> — where teacher forcing was introduced and named.
- Bengio et al. (2015) — **Scheduled Sampling for Sequence Prediction with RNNs**
  <https://arxiv.org/abs/1506.03099> — the best-known attempt to fix exposure bias.

In [14]:
import numpy as np

# ids for "<bos> a dog <eos>"  (1=<bos>, 2=<eos>, others=content)
tokens = np.array([1, 5, 6, 7, 2])
inputs  = tokens[:-1]      # everything but the last
targets = tokens[1:]       # everything but the first (shifted by one)
for i, t in zip(inputs, targets):
    print(f"input id {i}  ->  predict next id {t}")

input id 1  ->  predict next id 5
input id 5  ->  predict next id 6
input id 6  ->  predict next id 7
input id 7  ->  predict next id 2


## Positional embeddings

### 1. Which problem are we solving?

Embeddings (previous topic) gave every token a vector encoding **what** it is. Nothing so far
encodes **where** it is — and the model we're heading toward has no way to recover it.

An RNN never had this problem: it consumes tokens one at a time, so order is baked into the
computation itself. Attention (Step 4) has no such loop. It computes `softmax(QKᵀ/√d)V`, and
there is no `t` anywhere in that formula — every position is treated identically, so attention is
**permutation-equivariant**: shuffle the input rows and the output rows shuffle with them,
otherwise unchanged. To attention, a sentence is a **bag of tokens**.

Concretely, without position information *"dog bites man"* and *"man bites dog"* produce exactly
the same set of representations. Word order — most of syntax — is invisible.

**The purpose of positional embeddings:** inject "where" into the input, so each position's vector
carries both its identity and its place in the sequence, and attention can act on both. It is the
price attention pays for giving up recurrence.

### 2. Architecture and the math behind it

The mechanism is one line: give each **position** a vector, and **add** it to the token vector.

$$x_t = E_{\text{token}}[\text{id}_t] \;+\; E_{\text{pos}}[t]$$

- `id_t` — the token id at position `t`; `E_token[id_t]` is **what** the token is.
- `t` — the position index `0, 1, 2, …`; `E_pos[t]` is **where** it sits.
- `x_t` — the sum, and the actual input to the first Transformer block. Both terms live in the
  **same `d`-dimensional space**, which is what allows them to be added at all.
- `E_pos` is indexed by **position, not by token**: the same word gets a different vector at a
  different place, and different words at the same place share the same positional term.

**Why add instead of concatenate?** Concatenating is the obvious alternative and it works, but
addition wins on cost: width stays `d` instead of `d + d_pos`, so every downstream weight matrix
keeps its size. It looks lossy — two vectors crushed into one — but with `d` dimensions available
the network learns to hold "what" and "where" in largely separate subspaces and read them out with
different projections.

**Scheme 1 — learned.** Just another embedding table, identical in form to the token table:

$$E_{\text{pos}} \in \mathbb{R}^{(\text{max\_len} \times d)}, \qquad \text{row } t \text{ trainable}$$

- **`max_len` is a hard limit** — there is no row for position `max_len`, so the model cannot run
  on a longer sequence. This is one concrete origin of a model's **context window**.
- Backward is the **scatter-add** from the previous topic, but over *positions*: row `t` collects
  the upstream gradient from position `t` of **every sequence in the batch**.
- What GPT-2 used, and what we'll use: simplest to implement, and our sequences are short.

**Scheme 2 — sinusoidal** (the original Transformer). No parameters; the vector is a fixed
function of position:

$$PE_{(t,\,2i)} = \sin\!\left(\frac{t}{10000^{2i/d}}\right), \qquad
PE_{(t,\,2i+1)} = \cos\!\left(\frac{t}{10000^{2i/d}}\right)$$

- `t` — position; `i` — the dimension-pair index `0 … d/2−1`. **Even dimensions take sine, odd
  take cosine**, so dimensions come in `(sin, cos)` pairs sharing one frequency.
- `10000^{2i/d}` — the wavelength, growing **geometrically** with `i`: fast oscillation in early
  dimensions, very slow in late ones. Position is encoded at many scales at once, like the digits
  of a binary counter — low dimensions flip quickly, high ones separate coarse regions.
- `10000` — an arbitrary constant setting the longest wavelength (`2π·10000`); it only has to
  exceed the lengths you care about.
- **Why `sin`/`cos` pairs:** a shift of `k` positions is a **rotation** of each pair, so `PE(t+k)`
  is a *fixed linear function* of `PE(t)` — the same matrix at every `t`. That makes **relative**
  offsets linearly recoverable, which absolute learned vectors don't guarantee. (Verified in code
  below.)
- Being a formula, it is defined for **any** `t` — no `max_len` — so it can in principle
  extrapolate past the training length. In practice, not very well.

**Scheme 3 — RoPE (rotary)**, the modern default. Instead of adding anything to the input, it
**rotates `Q` and `K`** by an angle proportional to position, inside attention. The dot product
`q_m · k_n` then depends only on the **offset `m − n`**, making relative position intrinsic rather
than inferred. It needs Q/K to make sense, so it properly belongs to Step 4.

**The pipeline of calculations, in two scenarios.**

**(A) Learning (training).**

- The batch has a known length `T`, so the positions are simply `0 … T−1` — identical for every
  sequence in the batch.
- Look up (or compute) `E_pos[0…T−1]`, shape `(T, d)`, and **broadcast-add** it to the token
  embeddings of the whole batch. One add, no loop.
- Forward through the blocks, compute the loss as usual.
- Backward: gradient arriving at `x_t` passes **unchanged to both parents** (addition is a
  gradient splitter). The token table gets a scatter-add by token id; the position table gets a
  scatter-add by position index, summed over the batch.
- Sinusoidal takes no gradient step at all — it is a constant.

**(B) Actual working (generation).**

- Track how many tokens exist so far; the new token's position is that count.
- Add `E_pos[position]` to its token embedding before it enters the stack — one row, not a table.
- With a KV-cache, earlier positions keep the positional term they were given; nothing is
  recomputed.
- **The failure mode:** with a learned table, once `position ≥ max_len` there is no row and
  generation must stop or be truncated. Sinusoidal and RoPE can still produce a vector, though
  quality degrades beyond the trained range.

### 3. Applications (when to use)

Required in **every** Transformer, and anywhere a set-based model must know order:

- **Language models** — GPT (learned), the original Transformer (sinusoidal), LLaMA and most
  modern LLMs (RoPE).
- **Vision (ViT, Step 6)** — patches have **2-D** positions; a raster-ordered 1-D embedding works
  in practice, and 2-D variants encode row and column separately.
- **Our captioner** — the sequence is `[image patches | caption tokens]`, so positional embeddings
  let the model tell caption token 1 from token 3 *and* mark where the image block ends. A
  **segment/type embedding** (image vs. text) is the usual companion, added the same way.
- **Not needed** for RNNs, LSTMs or CNNs — recurrence and convolution already encode order or
  locality.

### 4. Advantages

- **Restores order to a set operation** — without it attention cannot represent syntax at all.
- **Essentially free** — one addition per token; no weight shape changes anywhere.
- **Composable** — because it is just a vector added to the input, other "where" signals stack the
  same way: segment/type, 2-D row and column, modality tags.
- **Sinusoidal costs zero parameters** and is defined at every position.
- **Learned costs nothing to reason about** — the same embedding layer you already wrote, with the
  same scatter-add backward.

### 5. Disadvantages

- **Learned tables impose a hard `max_len`** — no row, no inference. Extending context means
  retraining or interpolating the table.
- **Poor extrapolation** — sinusoidal is *defined* beyond the training length but degrades there;
  learned is undefined. A live research problem, not a solved one.
- **Absolute, not relative** — what usually matters is the *distance* between two tokens, and
  absolute encodings force the model to infer it. RoPE and ALiBi attack this directly.
- **Addition is an entangled channel** — "what" and "where" share the same `d` dimensions and can
  interfere; the separation is learned, not guaranteed.
- **1-D by default** — a plain positional embedding over raster-ordered image patches discards the
  2-D geometry (relevant in Step 6).

### 6. Useful resources

- **Original paper (sinusoidal + learned):** Vaswani et al. (2017) — **Attention Is All You Need**
  <https://arxiv.org/abs/1706.03762>, §3.5 — introduces positional encoding, gives the formula
  above, and reports that fixed sinusoids and learned embeddings perform about the same.
- **Learned positional embeddings** appeared earlier in Gehring et al. (2017) — **Convolutional
  Sequence to Sequence Learning** <https://arxiv.org/abs/1705.03122>.
- **RoPE — original paper:** Su et al. (2021) — **RoFormer: Enhanced Transformer with Rotary
  Position Embedding** <https://arxiv.org/abs/2104.09864> — what most modern LLMs use.
- **ALiBi — original paper:** Press, Smith & Lewis (2021) — **Train Short, Test Long: Attention
  with Linear Biases Enables Input Length Extrapolation** <https://arxiv.org/abs/2108.12409> —
  drops positional embeddings entirely and biases attention scores by distance instead.

In [15]:
import numpy as np

def sinusoidal_positions(max_len: int, d: int) -> np.ndarray:
    """Fixed positional encoding: PE[t, 2i] = sin(t / 10000^(2i/d)); odd dims use cos."""
    position = np.arange(max_len)[:, None]                       # (max_len, 1)
    dim_index = np.arange(d)[None, :]                            # (1, d)
    angle = position / (10000 ** (2 * (dim_index // 2) / d))     # each (sin,cos) pair shares one
    pe = np.zeros((max_len, d))
    pe[:, 0::2] = np.sin(angle[:, 0::2])                         # even dims: sine
    pe[:, 1::2] = np.cos(angle[:, 1::2])                         # odd dims: cosine
    return pe

pe = sinusoidal_positions(max_len=6, d=8)
print("positional embedding shape:", pe.shape)
print("position 0:", np.round(pe[0], 2))
print("position 1:", np.round(pe[1], 2))
# usage: x = token_embeddings + pe[:sequence_length]

# --- wavelengths grow geometrically: early dims oscillate fast, late dims slowly ---
long_pe = sinusoidal_positions(max_len=100, d=64)
print("\nfirst 6 positions, one fast dimension vs one slow dimension:")
print("  dim 0  (fast):", np.round(long_pe[:6, 0], 3), "...")
print("  dim 62 (slow):", np.round(long_pe[:6, 62], 5), "...")

# --- the shift property: PE(t+k) is a FIXED linear map of PE(t), identical at every t ---
def rotation_for_shift(shift: int, frequency: float) -> np.ndarray:
    angle = frequency * shift
    return np.array([[np.cos(angle),  np.sin(angle)],
                     [-np.sin(angle), np.cos(angle)]])

d, shift = 8, 3
frequency_of_first_pair = 1 / (10000 ** (0 / d))                 # i = 0 -> the fastest pair
rotation = rotation_for_shift(shift, frequency_of_first_pair)
pe_long = sinusoidal_positions(max_len=20, d=d)
holds_everywhere = all(np.allclose(rotation @ pe_long[t, 0:2], pe_long[t + shift, 0:2])
                       for t in range(10))
print("\nPE(t+3) = R @ PE(t) with the SAME matrix R at every t:", holds_everywhere)
print("-> relative offsets are linearly recoverable; that is why sin/cos come in pairs.")

# --- WHY THIS IS NEEDED: attention alone cannot tell the two orderings apart ---
def softmax_rows(scores: np.ndarray) -> np.ndarray:
    shifted = scores - scores.max(axis=-1, keepdims=True)
    exponentiated = np.exp(shifted)
    return exponentiated / exponentiated.sum(axis=-1, keepdims=True)

def self_attention(x: np.ndarray) -> np.ndarray:
    return softmax_rows(x @ x.T / np.sqrt(x.shape[-1])) @ x      # Q = K = V = x, simplest case

rng = np.random.default_rng(0)
vocabulary = rng.standard_normal((3, 8))                         # 0="dog", 1="bites", 2="man"
dog_bites_man = vocabulary[[0, 1, 2]]
man_bites_dog = vocabulary[[2, 1, 0]]

without_a = self_attention(dog_bites_man)
without_b = self_attention(man_bites_dog)
print("\nWITHOUT positional embeddings -- same representations, merely reordered:",
      np.allclose(without_a, without_b[[2, 1, 0]]))

positions = sinusoidal_positions(max_len=3, d=8)
with_a = self_attention(dog_bites_man + positions)
with_b = self_attention(man_bites_dog + positions)
print("WITH positional embeddings -- genuinely different:",
      not np.allclose(with_a, with_b[[2, 1, 0]]))
print("  max difference:", round(float(np.abs(with_a - with_b[[2, 1, 0]]).max()), 3))

positional embedding shape: (6, 8)
position 0: [0. 1. 0. 1. 0. 1. 0. 1.]
position 1: [0.84 0.54 0.1  1.   0.01 1.   0.   1.  ]

first 6 positions, one fast dimension vs one slow dimension:
  dim 0  (fast): [ 0.     0.841  0.909  0.141 -0.757 -0.959] ...
  dim 62 (slow): [0.      0.00013 0.00027 0.0004  0.00053 0.00067] ...

PE(t+3) = R @ PE(t) with the SAME matrix R at every t: True
-> relative offsets are linearly recoverable; that is why sin/cos come in pairs.

WITHOUT positional embeddings -- same representations, merely reordered: True
WITH positional embeddings -- genuinely different: True
  max difference: 1.073


## Sampling strategies at generation (good to know)

Given the next-token distribution, how do we pick? The choice trades **coherence** vs.
**diversity**:

- **Greedy** — argmax every step. Deterministic; safe but repetitive/generic.
- **Temperature `T`** — divide logits by `T` before softmax. `T<1` sharpens (more confident),
  `T>1` flattens (more random), `T->0` = greedy.
- **Top-k** — keep only the `k` highest-probability tokens, renormalize, sample. Cuts the long
  tail of nonsense.
- **Top-p (nucleus)** — keep the smallest set of tokens whose cumulative probability >= `p`
  (adaptive size), renormalize, sample.
- **Beam search** — keep the `b` best partial sequences by total probability; better for tasks
  with a "right" answer (translation), less used for open generation.

**Why it matters for us:** captions come from sampling — greedy for a stable demo,
temperature/top-k to show diversity in the report.

**Useful resources**

- **Top-k — original paper:** Fan, Lewis & Dauphin (2018) — **Hierarchical Neural Story
  Generation** <https://arxiv.org/abs/1805.04833>, §5.
- **Top-p / nucleus — original paper:** Holtzman et al. (2019) — **The Curious Case of Neural Text
  Degeneration** <https://arxiv.org/abs/1904.09751> — also the clearest explanation of *why*
  greedy decoding produces repetitive text.

In [16]:
import numpy as np

def softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - x.max()); return e / e.sum()

logits = np.array([2.0, 1.0, 0.5, -1.0, -2.0])

print("greedy pick:", int(np.argmax(logits)))
for T in [0.5, 1.0, 2.0]:
    print(f"T={T}: {np.round(softmax(logits / T), 3)}")   # temperature sharpen/flatten

def top_k_probs(logits: np.ndarray, k: int) -> np.ndarray:
    keep = np.argsort(logits)[::-1][:k]            # indices of the k largest
    masked = np.full_like(logits, -np.inf)
    masked[keep] = logits[keep]
    return softmax(masked)

print("top-2 probs:", np.round(top_k_probs(logits, 2), 3))

greedy pick: 0
T=0.5: [0.842 0.114 0.042 0.002 0.   ]
T=1.0: [0.603 0.222 0.134 0.03  0.011]
T=2.0: [0.41  0.249 0.194 0.092 0.056]
top-2 probs: [0.731 0.269 0.    0.    0.   ]


# Step 4 — Sequence Models: RNN → Attention

Step 3 gave us tokens, embeddings and the next-token objective, but the only model we had was
the **bigram** — context of exactly one token. This step is about models that actually *use*
history. We start with the **RNN** (the pre-2017 answer), understand precisely where it breaks,
and that failure is what motivates **attention**.

## Recurrent Neural Networks (RNN)

### 1. Which problem are we solving?

Coming from Step 3, our language model was a **bigram**: `p(w_t | w_{t-1})`. Two hard limits:

- **Fixed, tiny context.** An n-gram sees only the last `n-1` tokens. To predict the last word
  of *"the dog that chased the cat across the yard was ___"*, you need the subject from 9 tokens
  back. Extending `n` doesn't scale: the count table grows as `vocab^n`.
- **No parameter sharing across time.** A plain MLP over a fixed window learns a *separate*
  weight for "the token 3 positions ago". It can't reuse "what a verb looks like" at every
  position, and it can't accept a sequence of a length it never saw.

We want a model that (a) handles **variable-length** sequences, (b) carries information from
**arbitrarily far back**, and (c) uses the **same parameters at every time step**. That model is
the RNN: keep a **hidden state** — a running summary of everything seen so far — and update it
with one shared function, once per token.

$$\text{bigram: } p(w_t \mid w_{t-1}) \qquad\longrightarrow\qquad \text{RNN: } p(w_t \mid h_{t-1}), \quad h_{t-1} = f(w_{<t})$$

The whole prefix is compressed into one vector `h`. That compression is the RNN's power *and*,
as we'll see in §5, exactly its weakness.

### 2. Architecture and the math behind it

An RNN is a **loop**: one small network applied repeatedly, its output fed back as input.

```
        x₁          x₂          x₃            <- input token embeddings
        │           │           │
h₀ ──▶ [A] ──h₁──▶ [A] ──h₂──▶ [A] ──h₃──▶    <- SAME weights A at every step
        │           │           │
        y₁          y₂          y₃            <- per-step output (e.g. next-token logits)
```

Unrolled, it's a deep feedforward net whose depth equals the **sequence length** — and whose
layers all **share one set of weights**.

**The recurrence (hidden state update):**

$$h_t = \tanh\!\big(W_{xh}\,x_t \;+\; W_{hh}\,h_{t-1} \;+\; b_h\big)$$

- `x_t` ∈ ℝ^`d_in` — input at step `t` (for us: the token embedding from Step 3).
- `h_{t-1}` ∈ ℝ^`d_h` — hidden state from the previous step: **the memory**, a compressed
  summary of tokens `1..t-1`. Initialized `h_0 = 0`.
- `W_xh` ∈ ℝ^(`d_h × d_in`) — **input-to-hidden**: how much the *new* token influences memory.
- `W_hh` ∈ ℝ^(`d_h × d_h`) — **hidden-to-hidden**, the recurrent matrix: how the old memory is
  transformed/retained. This single matrix is applied `T` times — the source of both long-range
  memory and the vanishing/exploding problem (§5).
- `b_h` ∈ ℝ^`d_h` — bias.
- `tanh` — the nonlinearity; **bounds `h` to (−1, 1)**, which keeps the repeatedly-multiplied
  state from blowing up. Its derivative `1 − h²` is ≤ 1 everywhere — remember this for §5.
- **`t` does not appear in the weights**: the same `W_xh, W_hh, b_h` are reused at every step
  (**weight tying across time**). That's what makes variable length possible.

**The output head (per step):**

$$y_t = W_{hy}\,h_t + b_y \qquad\qquad p_t = \text{softmax}(y_t)$$

- `W_hy` ∈ ℝ^(`vocab × d_h`) — projects the hidden state to **logits** over the vocabulary.
- `softmax` → next-token distribution; loss is **cross-entropy** vs. the true next token
  (Step 2), averaged over all `T` positions:

$$L = \frac{1}{T}\sum_{t=1}^{T} -\log p_t[w_{t+1}]$$

**Training: Backpropagation Through Time (BPTT).** Unroll the loop, then run ordinary backprop
on the resulting deep net. Two consequences of weight sharing:

$$\frac{\partial L}{\partial W_{hh}} \;=\; \sum_{t=1}^{T} \frac{\partial L}{\partial h_t}\,\frac{\partial h_t}{\partial W_{hh}}$$

- The gradient of a shared weight is the **sum of its contributions at every time step** (same
  accumulate-on-reuse rule as the embedding scatter-add in Step 3).
- `∂L/∂h_t` has **two** sources: the loss at step `t` (through `W_hy`) **plus** the gradient
  flowing back from step `t+1` (through `W_hh`) — memory is used later, so blame arrives from
  later.

And the term that decides whether long-range learning works at all — the gradient travelling
from step `t` back to an earlier step `k`:

$$\frac{\partial h_t}{\partial h_k} \;=\; \prod_{i=k+1}^{t} \frac{\partial h_i}{\partial h_{i-1}} \;=\; \prod_{i=k+1}^{t} \operatorname{diag}\!\big(1 - h_i^2\big)\, W_{hh}$$

- It is a **product of `t − k` Jacobians** — one factor per step of distance.
- `diag(1 − h_i²)` — the `tanh` derivative, each entry in **(0, 1]** (and ≈0 when `h` saturates
  near ±1).
- `W_hh` appears **`t − k` times**, so its largest singular value `σ` is raised to that power:
  `σ < 1` → the product **decays exponentially** (*vanishing gradient* — the model literally
  cannot learn the dependency); `σ > 1` → it **explodes** to `inf`/`nan`.
- Exponential in **distance**: this is the precise, mathematical reason a plain RNN forgets.
  (Compare: attention connects position `t` to position `k` in **one** step — a path length of
  1 instead of `t − k`. That is the punchline of the next section.)

**Practical fixes** used with RNNs: **gradient clipping** (rescale the gradient if its norm
exceeds a threshold — cures exploding), **truncated BPTT** (backprop only `k` steps back to bound
cost), and gated cells — **LSTM/GRU** — which add an *additive* memory path (`c_t = f_t ⊙ c_{t-1}
+ i_t ⊙ g_t`) so gradients can flow across many steps without being multiplied by `W_hh` each
time. Gates mitigate vanishing; they don't remove the sequential bottleneck.

**The pipeline of calculations, in two scenarios.** Same recurrence both times — what differs
is what flows through it.

**(A) Learning (training).** The whole sequence is known upfront (teacher forcing):

- Take the caption, split it into `inputs` and `targets` shifted by one.
- Embed the input ids into `x_1 … x_T`.
- Run the loop forward, `t = 1 … T`: compute `h_t` from `x_t` and `h_{t-1}`, then the logits
  `y_t`. Keep every `x_t` and `h_t` — the backward pass needs them.
- Compute the loss: cross-entropy of `y_t` against `target_t`, averaged over all `T` positions.
- Run the loop backward, `t = T … 1`. Each step's hidden state gets blame from **two** places:
  its own output, and the step after it (memory is used later, so blame comes from later).
- Accumulate the gradients of the shared weights — every step adds into the *same* `W_xh`,
  `W_hh`, `W_hy` (this summing *is* weight sharing).
- Clip the gradient if its norm is too large, then take one Adam step. Repeat on the next batch.

**(B) Actual working (generation).** Nothing exists yet; the model writes it:

- Weights are frozen. No targets, no loss, no backward pass, nothing cached.
- Start with `h = 0` (for captioning: `h` = the image feature vector — that's *Show and Tell*)
  and feed `<bos>`.
- Compute the new `h`, overwriting the old one — it's no longer needed.
- Project `h` to logits, softmax to probabilities, pick a token (greedy / temperature / top-k).
- Emit that token and feed it **back in** as the next input.
- Repeat until `<eos>` or a length cap.

**The differences that matter:**

- **Input:** training feeds ground-truth tokens; generation feeds the model's own last output.
  That mismatch is **exposure bias** (Step 3).
- **Passes:** training is forward + backward; generation is forward only.
- **Memory:** training stores all `T` hidden states (why *truncated* BPTT exists); generation
  stores exactly one vector, no matter how long the output — the one place a plain RNN still
  beats attention, whose KV-cache keeps growing.
- **Parallelism:** neither can be parallelized across time — `h_t` needs `h_{t-1}`. Training a
  length-`T` sequence means `T` dependent steps. This is the bottleneck attention removes.

### 3. Applications (when to use)

Anything where the input is a **sequence** and order matters:

- **Language modeling / text generation** (char-RNN — the direct ancestor of what we're building)
- **Sequence classification**: sentiment, intent — read the sequence, classify from the final `h`
- **Sequence labeling**: POS tagging, NER — one output per step
- **Seq2seq**: translation, summarization (encoder RNN → decoder RNN; the 2014-2016 standard,
  and where attention was *invented* as a patch)
- **Time series & audio**: forecasting, speech recognition — where signals are naturally streamed
- **Image captioning** — *our task*: the classic **Show and Tell** (2015) fed a CNN image vector
  as `h_0` into an LSTM decoder. We do the same job with a Transformer instead.

**Still genuinely useful today** when: the sequence is **very long or unbounded** (streaming
audio/sensors — `O(1)` memory per step beats attention's `O(n²)`), latency per token must be
tiny on-device, or the dataset is small. Otherwise, Transformers win.

### 4. Advantages

- **Variable-length input** — the loop just runs longer; no fixed window.
- **Constant parameter count** — independent of sequence length (weights are shared across time).
- **In principle unbounded context** — `h_t` can carry information from any earlier step.
- **`O(1)` memory / `O(n)` compute** in sequence length — cheap and **streaming-friendly**:
  process a token, update `h`, discard the token.
- **Inductive bias for order** — recency and sequence structure are built in; no positional
  encoding needed (contrast Step 3's positional embeddings, which attention *requires*).

### 5. Disadvantages

- **Vanishing / exploding gradients** — the `∏ W_hh` product of §2. Learning dependencies beyond
  ~10-20 steps is unreliable; LSTM/GRU push this out but don't solve it.
- **Inherently sequential — cannot be parallelized across time.** `h_t` needs `h_{t-1}`, so
  training on a length-`T` sequence takes `T` dependent steps and can't use a GPU's parallelism.
  **This, not accuracy, is the main reason Transformers replaced RNNs**: attention computes all
  positions at once.
- **Information bottleneck** — the entire prefix must fit in one fixed-size vector `h`. In
  seq2seq, the whole source sentence squeezed into one vector was the failure that attention was
  invented to fix.
- **Recency bias** — recent tokens dominate `h`; distant ones fade.
- **No direct access to the past** — to use token `k` at step `t`, the information must survive
  `t − k` lossy overwrites. Attention *reads token `k` directly*.
- **Hard to interpret** — no equivalent of an attention map showing what the model looked at.

> **The bridge to Step 4's real content:** every disadvantage above is about the *sequential,
> compressed* path between positions. Attention replaces it with a **direct, parallel, weighted
> lookup** over all positions at once — same goal (use the context), opposite mechanism.

### 6. Useful resources

- **Original paper:** Elman (1990) — **Finding Structure in Time**, *Cognitive Science* 14(2),
  179-211 <https://onlinelibrary.wiley.com/doi/10.1207/s15516709cog1402_1> — the "Elman network",
  the recurrent architecture in §2 above. (Jordan, 1986, is the closely related predecessor.)
- Pascanu, Mikolov & Bengio (2013) — **On the difficulty of training Recurrent Neural Networks**
  <https://arxiv.org/abs/1211.5063> — the formal treatment of §2's gradient product, and the
  origin of gradient clipping.
- Karpathy — **The Unreasonable Effectiveness of Recurrent Neural Networks**
  <http://karpathy.github.io/2015/05/21/rnn-effectiveness/> — the char-RNN post: the same
  architecture as above, ~100 lines of NumPy, trained on Shakespeare / C code / LaTeX. Read the
  generated samples, then his `min-char-rnn.py` gist for a full forward + BPTT in one file.
- Olah — **Understanding LSTM Networks** <https://colah.github.io/posts/2015-08-Understanding-LSTMs/>
  — the canonical picture of why gates fix the vanishing product in §2.
- Bahdanau et al. (2014) — *Neural Machine Translation by Jointly Learning to Align and Translate*
  — attention invented as a patch for the RNN bottleneck (§5). The direct prequel to Step 4.

In [17]:
import numpy as np

# Minimal RNN: forward over a sequence, one shared set of weights reused at every step.
class SimpleRNN:
    def __init__(self, input_size: int, hidden_size: int, output_size: int, seed: int = 0) -> None:
        rng = np.random.default_rng(seed)
        self.hidden_size: int = hidden_size
        # Xavier-ish scaling (Step 1) so the repeated multiplications stay well behaved
        self.input_to_hidden: np.ndarray  = rng.standard_normal((hidden_size, input_size))  / np.sqrt(input_size)   # W_xh
        self.hidden_to_hidden: np.ndarray = rng.standard_normal((hidden_size, hidden_size)) / np.sqrt(hidden_size)  # W_hh
        self.hidden_bias: np.ndarray     = np.zeros(hidden_size)                                                  # b_h
        self.hidden_to_output: np.ndarray = rng.standard_normal((output_size, hidden_size)) / np.sqrt(hidden_size)   # W_hy
        self.output_bias: np.ndarray     = np.zeros(output_size)                                                  # b_y

    def forward(self, inputs: np.ndarray) -> tuple:
        # inputs: (T, input_size) -- one embedding per time step
        sequence_length: int = inputs.shape[0]
        hidden_state: np.ndarray = np.zeros(self.hidden_size)      # h_0 = 0
        hidden_states: list = []
        logits: list = []
        for t in range(sequence_length):
            # h_t = tanh(W_xh x_t + W_hh h_{t-1} + b_h)   <-- the recurrence
            hidden_state = np.tanh(
                self.input_to_hidden @ inputs[t] + self.hidden_to_hidden @ hidden_state + self.hidden_bias
            )
            hidden_states.append(hidden_state)
            # y_t = W_hy h_t + b_y                        <-- per-step output head
            logits.append(self.hidden_to_output @ hidden_state + self.output_bias)
        return np.array(hidden_states), np.array(logits)

rng = np.random.default_rng(1)
sequence_length, input_size, hidden_size, vocab_size = 5, 4, 3, 6
inputs = rng.standard_normal((sequence_length, input_size))       # pretend token embeddings

rnn = SimpleRNN(input_size, hidden_size, vocab_size)
hidden_states, logits = rnn.forward(inputs)

print("hidden states shape:", hidden_states.shape, "(T, hidden_size)")
print("logits shape       :", logits.shape, "(T, vocab_size)")
print("h_1:", np.round(hidden_states[0], 3))
print("h_5:", np.round(hidden_states[-1], 3), " <- summary of ALL 5 tokens")

# Parameter count does NOT depend on sequence length -- same net runs on any T
_, logits_long = rnn.forward(rng.standard_normal((50, input_size)))
print("same weights on T=50:", logits_long.shape)

hidden states shape: (5, 3) (T, hidden_size)
logits shape       : (5, 6) (T, vocab_size)
h_1: [ 0.005 -0.333 -0.648]
h_5: [-0.634 -0.702  0.441]  <- summary of ALL 5 tokens
same weights on T=50: (50, 6)


In [18]:
import numpy as np

# Vanishing / exploding gradients, empirically: how strongly does h_T still depend on h_k?
# Magnitude of  prod_{i=k+1..T} diag(1 - h_i^2) W_hh  as the distance (T - k) grows.
def gradient_norm_over_distance(spectral_scale: float, hidden_size: int = 20,
                             sequence_length: int = 40, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    recurrent = rng.standard_normal((hidden_size, hidden_size)) / np.sqrt(hidden_size)
    recurrent *= spectral_scale / max(abs(np.linalg.eigvals(recurrent)))   # set largest |eigenvalue|
    inputs = rng.standard_normal((sequence_length, hidden_size)) * 0.5

    hidden = np.zeros(hidden_size)
    hiddens = []
    for t in range(sequence_length):                    # forward pass, collect h_t
        hidden = np.tanh(recurrent @ hidden + inputs[t])
        hiddens.append(hidden)

    jacobian = np.eye(hidden_size)                      # start from dh_T/dh_T = I
    norms = []
    for t in reversed(range(sequence_length)):          # walk backward, accumulating the product
        jacobian = jacobian @ (np.diag(1 - hiddens[t]**2) @ recurrent)
        norms.append(np.linalg.norm(jacobian))
    return np.array(norms)                             # index = distance back in time

print("gradient norm  d(h_T)/d(h_k)  by distance (T - k):")
for scale, label in [(0.5, "sigma=0.5 -> VANISHING"),
                     (1.0, "sigma=1.0 -> slow decay"),
                     (2.5, "sigma=2.5 -> EXPLODING")]:
    norms = gradient_norm_over_distance(scale)
    print(f"  {label:26s} 1: {norms[0]:9.2e}   10: {norms[9]:9.2e}   40: {norms[39]:9.2e}")

# Note: even sigma slightly > 1 often still decays -- the diag(1 - h^2) factors are < 1 when
# tanh saturates, so saturation damps growth. It takes a clearly large sigma to truly explode.
print("\nExponential in DISTANCE -> a plain RNN cannot learn long-range dependencies.")
print("Attention connects any two positions in ONE step (path length 1, no product at all).")

gradient norm  d(h_T)/d(h_k)  by distance (T - k):
  sigma=0.5 -> VANISHING     1:  1.74e+00   10:  3.99e-04   40:  8.60e-16
  sigma=1.0 -> slow decay    1:  3.45e+00   10:  2.15e-01   40:  1.41e-05
  sigma=2.5 -> EXPLODING     1:  5.99e+00   10:  3.52e+01   40:  2.76e+03

Exponential in DISTANCE -> a plain RNN cannot learn long-range dependencies.
Attention connects any two positions in ONE step (path length 1, no product at all).


## LSTM (Long Short-Term Memory)

### 1. Which problem are we solving?

The RNN above works, right up until the distance matters. Its gradient across `t − k` steps was

$$\frac{\partial h_t}{\partial h_k} = \prod_{i=k+1}^{t} \operatorname{diag}(1 - h_i^2)\, W_{hh}$$

a **product of `t − k` factors fixed by the weights**. Two problems follow from that single line:

- **The gradient decays exponentially** in distance, so dependencies beyond ~10-20 steps are
  effectively unlearnable. Olah's example: *"the clouds are in the ___"* is easy (the answer is
  two words back), but *"I grew up in France … I speak fluent ___"* needs information from
  dozens of steps back — and the gradient that would teach the model to use it has already died.
- **Memory is rewritten every step, unconditionally.** `h_{t-1}` goes through `W_hh` and `tanh`
  to become `h_t`. There is no way for the network to say *"leave this alone for a while"* —
  after 30 steps, anything worth keeping has been through 30 lossy transformations.

The LSTM (Hochreiter & Schmidhuber, 1997) fixes both with one idea: add a **second memory** that
is updated by **addition instead of matrix multiplication**, and let the network **learn, per step
and per dimension, what to erase, what to write, and what to read out**.

### 2. Architecture and the math behind it

An LSTM step carries **two** vectors forward, not one:

- **`C_t` — the cell state.** The long-term memory. Private: no other part of the network reads
  it directly. This is Olah's "conveyor belt" running straight through the top of the diagram.
- **`h_t` — the hidden state.** A *filtered view* of `C_t`. The short-term working output: it
  feeds the output head, and it feeds the next step's gate decisions.

Everything is controlled by three **gates**. A gate is a sigmoid layer producing a vector in
`(0, 1)`, used as an elementwise multiplier — a soft, differentiable valve. `0` = block
completely, `1` = pass through untouched.

All three gates read the same thing: the concatenation `[h_{t-1}, x_t]` (what I was thinking, plus
what I'm seeing now).

**Step 1 — forget gate: what to erase from memory.**

$$f_t = \sigma\big(W_f \cdot [h_{t-1}, x_t] + b_f\big)$$

- `σ` — sigmoid, squashing to `(0, 1)`; this is what makes `f_t` a valve rather than a value.
- `[h_{t-1}, x_t]` — concatenated previous hidden state and current input, length `d_h + d_in`.
- `W_f` ∈ ℝ^(`d_h × (d_h + d_in)`), `b_f` — the forget gate's own weights.
- `f_t` ∈ `(0,1)^{d_h}` — **one value per memory slot**. `f_t[j] ≈ 1` → keep slot `j` intact;
  `≈ 0` → wipe it. Olah's example: on seeing a new subject, forget the old subject's gender.
- **`b_f` is usually initialized positive** (e.g. `1.0`) so gates start near 1 and memory is
  retained by default at the beginning of training — a real, important practical trick.

**Step 2 — input gate + candidate: what to write.**

$$i_t = \sigma\big(W_i \cdot [h_{t-1}, x_t] + b_i\big) \qquad \tilde{C}_t = \tanh\big(W_C \cdot [h_{t-1}, x_t] + b_C\big)$$

- `i_t` ∈ `(0,1)^{d_h}` — **how much** to write into each slot (the write-enable line).
- `C̃_t` ∈ `(−1,1)^{d_h}` — **what** to write: the candidate content. `tanh`, not sigmoid,
  because content is a signed value, not a valve.
- The split matters: *how much* and *what* are decided by **separate** weights, so the network can
  compute a candidate and independently decide to ignore it.

**Step 3 — update the cell state. This is the important line.**

$$C_t = f_t \odot C_{t-1} \;+\; i_t \odot \tilde{C}_t$$

- `⊙` — **elementwise** product, so every slot is an independent little memory with its own
  erase/write decision. In one step the LSTM can forget slot 3, hold slot 7, overwrite slot 12.
- `f_t ⊙ C_{t-1}` — the retained part of the old memory.
- `i_t ⊙ C̃_t` — the newly written part.
- **What's absent is the point:** no weight matrix and no `tanh` touch `C_{t-1}`. If the network
  sets `f_t ≈ 1, i_t ≈ 0`, then `C_t = C_{t-1}` — an **exact, lossless copy**, and information
  rides the belt unchanged for hundreds of steps. The vanilla RNN cannot express this.
- **The gradient consequence:** the path from `C_t` back to `C_{t-1}` is just

$$\frac{\partial C_t}{\partial C_{t-1}} = \operatorname{diag}(f_t) \qquad\Longrightarrow\qquad \frac{\partial C_t}{\partial C_k} = \prod_{i=k+1}^{t} \operatorname{diag}(f_i)$$

  Still a product — but of **gate values the network controls**, not of a fixed `W_hh`. Hold the
  gates near 1 and the gradient arrives from 100 steps back essentially intact. This additive,
  gated highway is the *entire* reason LSTMs work. (It's the same trick as a residual connection,
  Step 5 — an unobstructed path for the gradient.)

**Step 4 — output gate: what to read out.**

$$o_t = \sigma\big(W_o \cdot [h_{t-1}, x_t] + b_o\big) \qquad h_t = o_t \odot \tanh(C_t)$$

- `tanh(C_t)` — squash the memory to `(−1,1)` before exposing it (`C` itself is unbounded, since
  it's built by accumulation).
- `o_t` — the **read-enable** line: which parts of memory are relevant *right now*.
- `h_t` is therefore only a **filtered view** of the memory. This is the second thing the cell
  state buys us: a fact can sit in `C` for 50 steps with `o_t ≈ 0` keeping it hidden, then be
  exposed at the exact moment it's needed. In a vanilla RNN, memory *is* the output — anything you
  want to remember must also be part of what you emit right now.
- The output head is then the same as the RNN's: `y_t = W_hy h_t + b_y`, softmax, cross-entropy.

**The three gates in one line each:**

| gate | formula | role | RAM analogy |
|---|---|---|---|
| `f_t` | `σ(W_f·[h_{t-1},x_t])` | what to drop from `C` | erase enable |
| `i_t`, `C̃_t` | `σ(...)`, `tanh(...)` | how much / what to store | write enable + data |
| `o_t` | `σ(W_o·[h_{t-1},x_t])` | what to expose as `h_t` | read enable |

So an LSTM is a small **differentiable memory with learned read/write/erase control**, where a
vanilla RNN is a single register that gets fully overwritten every step.

**Worked intuition.** Reading *"The **keys** that the man on the roof left ... **were** rusty"*:
one slot of `C` latches "subject is plural" at *keys* (`i≈1`), `f≈1` holds it through the whole
clause while `o≈0` keeps it out of the way, then `o` opens at *were* to get the agreement right.
A new subject appears → `f→0` for that slot → forgotten. Same story for Olah's France/French.

**The pipeline of calculations, in two scenarios.**

**(A) Learning (training).** Whole sequence known upfront (teacher forcing):

- Embed the input ids into `x_1 … x_T`; start with `h_0 = 0`, `C_0 = 0`.
- Forward loop `t = 1 … T`: from `[h_{t-1}, x_t]` compute the three gates and the candidate, then
  `C_t`, then `h_t`, then the logits `y_t`. Cache **all** of `f_t, i_t, o_t, C̃_t, C_t, h_t` —
  the backward pass needs every one (≈4× the RNN's activation memory).
- Loss: cross-entropy of `y_t` against `target_t`, averaged over all `T` positions.
- Backward loop `t = T … 1`. Each step's `C_t` collects gradient from **two** paths: through
  `h_t = o_t ⊙ tanh(C_t)` (this step's output) and straight from `C_{t+1}` scaled by `f_{t+1}` —
  the highway. The gates get gradients too, so the model *learns when to remember*.
- Accumulate into the shared weights (`W_f, W_i, W_C, W_o` and biases) — one `+=` per time step,
  same weight-sharing rule as the RNN.
- Clip the gradient norm (exploding is still possible through the gates), then one Adam step.

**(B) Actual working (generation).** Weights frozen, nothing exists yet:

- Carry **two** vectors instead of one: `h` and `C`, both starting at 0 (for captioning, seed
  them from the image features).
- Feed `<bos>`. Compute the gates from `[h, x]`, update `C` in place, then `h` from the new `C`.
- Project `h` to logits → softmax → pick a token (greedy / temperature / top-k).
- Emit it, feed it back in as the next input, repeat until `<eos>` or a length cap.
- No loss, no backward, nothing cached — only the current `(h, C)` pair.

**What differs from the RNN pipeline:** 4× the parameters and 4× the cached activations per step
(four weight matrices instead of one), two state vectors to carry instead of one, and a gradient
path that survives distance. The **sequential** structure is completely unchanged — that's the
part attention has to fix.

**Variants (from the post).**

- **Peephole connections** — let the gates also look at `C_{t-1}`, not just `[h_{t-1}, x_t]`, so
  memory content can influence the control decisions.
- **Coupled forget/input** — set `i_t = 1 − f_t`: only write to a slot when you erase it.
- **GRU** (Cho et al., 2014) — the popular simplification: merge forget+input into one **update
  gate** `z_t`, and **merge `C` and `h` into a single state**:

$$z_t = \sigma(W_z\cdot[h_{t-1},x_t]), \quad r_t = \sigma(W_r\cdot[h_{t-1},x_t])$$
$$\tilde{h}_t = \tanh\big(W\cdot[r_t \odot h_{t-1},\, x_t]\big), \qquad h_t = (1-z_t)\odot h_{t-1} + z_t \odot \tilde{h}_t$$

  - `z_t` — update gate: interpolate between keeping the old state and taking the new candidate.
  - `r_t` — reset gate: how much of the old state the candidate is even allowed to see.
  - The `(1−z)·h + z·h̃` form keeps the **additive** highway, which is what mattered. 3 weight
    matrices instead of 4, no separate `C`, usually comparable accuracy — pick either.

### 3. Applications (when to use)

Everywhere the RNN was used, but where the dependencies are actually long — which was, in
practice, everywhere. From ~2014 to ~2018 the LSTM was the default sequence model:

- **Language modeling and text generation** (Karpathy's char-RNN is an LSTM)
- **Machine translation** — seq2seq encoder/decoder LSTMs; Google Translate, 2016
- **Speech recognition**, handwriting recognition, OCR
- **Image captioning — our task's ancestor:** *Show and Tell* (2015) fed a CNN vector into an LSTM
  decoder; *Show, Attend and Tell* (2015) added attention on top of that LSTM
- **Time series**: forecasting, anomaly detection, sensor/ECG signals
- **Reinforcement learning** — an LSTM layer gives a policy memory of past observations

**Still a sound choice today** when the stream is unbounded or truly online (`O(1)` state per
token, no growing KV-cache), when latency/memory on-device is tight, or when the dataset is small
enough that a Transformer would just overfit. For anything you can batch and parallelize, use a
Transformer.

### 4. Advantages

- **Learns long-range dependencies** — the additive `C` path with `∂C_t/∂C_{t-1} = diag(f_t)`
  keeps gradients alive over hundreds of steps instead of tens.
- **Learned, per-dimension memory control** — the network decides what to keep, overwrite, and
  expose, separately for every slot and every step. No hand-designed window.
- **Separates storage from output** — a fact can be held without being emitted (`o_t ≈ 0`),
  impossible in a vanilla RNN.
- **Robust in practice** — trains reliably with modest tuning; a positive forget-gate bias is
  usually all the coaxing it needs.
- **Keeps every structural RNN advantage** — variable length, weight sharing across time, `O(1)`
  memory at generation, streaming-friendly, order built in (no positional encoding needed).

### 5. Disadvantages

- **Still strictly sequential.** `C_t` needs `C_{t-1}`. Training a length-`T` sequence is `T`
  dependent steps that a GPU cannot parallelize. **This is the reason Transformers replaced
  LSTMs** — attention computes all positions at once. Gates fixed the gradient, not the speed.
- **~4× the parameters and compute** of a vanilla RNN per step (four gate matrices), and ~4× the
  cached activations during BPTT.
- **Mitigates rather than eliminates vanishing.** `∏ f_i` still decays if the gates sit below 1 —
  a badly initialized forget gate (`f ≈ 0.5`) vanishes just as fast as a plain RNN. Exploding
  gradients also remain, so clipping is still standard.
- **The bottleneck survives.** However well `C` retains, the entire prefix is still squeezed into
  one fixed-size vector. In seq2seq this was the failure that **attention was invented to fix**.
- **Long-range in principle ≠ in practice.** Real LSTM LMs use a few hundred tokens of effective
  context, far short of a Transformer's window.
- **Hard to interpret** — no attention map. You can probe individual cell slots, but there's no
  direct readout of "what the model looked at."
- **Fiddly to implement** — four gates, two states, and a backward pass with two gradient paths
  into `C_t` is markedly more error-prone to hand-derive than attention's.

> **Where this leaves us.** The LSTM removed the *gradient* obstacle to long-range memory but kept
> the two structural ones: **sequential computation** and **one fixed-size summary vector**.
> Olah's own closing note points at the fix — let each step *"pick information to look at from
> some larger collection of information."* That is attention, and it's the next section.

### 6. Useful resources

- Olah — **Understanding LSTM Networks**
  <https://colah.github.io/posts/2015-08-Understanding-LSTMs/> — the source for this section; the
  four-step walkthrough with the conveyor-belt diagram is the clearest explanation there is.
- Karpathy — **The Unreasonable Effectiveness of RNNs**
  <http://karpathy.github.io/2015/05/21/rnn-effectiveness/> — char-LSTM in ~100 lines of NumPy,
  plus the cell-visualization section showing individual slots tracking quote-open/quote-closed.
- **Original paper:** Hochreiter & Schmidhuber (1997) — **Long Short-Term Memory**, *Neural
  Computation* 9(8), 1735-1780 <https://www.bioinf.jku.at/publications/older/2604.pdf>. Note the
  forget gate is *not* in it — Gers, Schmidhuber & Cummins (2000) added it later, and it turned
  out to be the most important gate of all.
- **GRU — original paper:** Cho et al. (2014) — **Learning Phrase Representations using RNN
  Encoder-Decoder for Statistical Machine Translation** <https://arxiv.org/abs/1406.1078>.
- Greff et al. (2015) — **LSTM: A Search Space Odyssey** <https://arxiv.org/abs/1503.04069> —
  ablates every gate; the empirical answer to "which parts actually matter" (the forget gate
  matters most).

In [19]:
import numpy as np

def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-x))

# One LSTM cell: carries TWO states forward -- cell_state (C) and hidden_state (h).
class LSTMCell:
    def __init__(self, input_size: int, hidden_size: int, forget_bias: float = 1.0, seed: int = 0) -> None:
        rng = np.random.default_rng(seed)
        self.hidden_size: int = hidden_size
        concat_size: int = hidden_size + input_size          # gates read [h_{t-1}, x_t]
        scale: float = 1 / np.sqrt(concat_size)

        self.forget_weights: np.ndarray    = rng.standard_normal((hidden_size, concat_size)) * scale  # W_f
        self.input_weights: np.ndarray     = rng.standard_normal((hidden_size, concat_size)) * scale  # W_i
        self.candidate_weights: np.ndarray = rng.standard_normal((hidden_size, concat_size)) * scale  # W_C
        self.output_weights: np.ndarray    = rng.standard_normal((hidden_size, concat_size)) * scale  # W_o

        # forget bias starts POSITIVE -> f_t near 1 -> memory retained by default early in training
        self.forget_bias: np.ndarray    = np.full(hidden_size, forget_bias)
        self.input_bias: np.ndarray     = np.zeros(hidden_size)
        self.candidate_bias: np.ndarray = np.zeros(hidden_size)
        self.output_bias: np.ndarray    = np.zeros(hidden_size)

    def step(self, x: np.ndarray, hidden_state: np.ndarray, cell_state: np.ndarray) -> tuple:
        concatenated = np.concatenate([hidden_state, x])                              # [h_{t-1}, x_t]

        forget_gate    = sigmoid(self.forget_weights    @ concatenated + self.forget_bias)     # f_t: what to erase
        input_gate     = sigmoid(self.input_weights     @ concatenated + self.input_bias)      # i_t: how much to write
        candidate     = np.tanh(self.candidate_weights @ concatenated + self.candidate_bias)  # C~_t: what to write
        output_gate    = sigmoid(self.output_weights    @ concatenated + self.output_bias)     # o_t: what to expose

        new_cell_state   = forget_gate * cell_state + input_gate * candidate   # C_t -- ADDITIVE, no matrix on C_{t-1}
        new_hidden_state = output_gate * np.tanh(new_cell_state)               # h_t -- filtered VIEW of the memory
        return new_hidden_state, new_cell_state, forget_gate

    def forward(self, inputs: np.ndarray) -> tuple:
        hidden_state = np.zeros(self.hidden_size)      # h_0 = 0
        cell_state   = np.zeros(self.hidden_size)      # C_0 = 0
        hidden_states, cell_states, forget_gates = [], [], []
        for t in range(inputs.shape[0]):
            hidden_state, cell_state, forget_gate = self.step(inputs[t], hidden_state, cell_state)
            hidden_states.append(hidden_state); cell_states.append(cell_state); forget_gates.append(forget_gate)
        return np.array(hidden_states), np.array(cell_states), np.array(forget_gates)

rng = np.random.default_rng(1)
sequence_length, input_size, hidden_size = 8, 4, 5
inputs = rng.standard_normal((sequence_length, input_size))

lstm = LSTMCell(input_size, hidden_size)
hidden_states, cell_states, forget_gates = lstm.forward(inputs)

print("hidden states h:", hidden_states.shape, " cell states C:", cell_states.shape)
print("\nmean forget gate per step (near 1 -> memory kept):", np.round(forget_gates.mean(axis=1), 3))
print("\nC grows by ACCUMULATION (unbounded), h is bounded by tanh * o:")
print("  |C| max over steps:", np.round(np.abs(cell_states).max(axis=1), 3))
print("  |h| max over steps:", np.round(np.abs(hidden_states).max(axis=1), 3))

# The gate that HOLDS: force f=1, i=0 and the cell state is copied EXACTLY
frozen = LSTMCell(input_size, hidden_size, forget_bias=20.0)     # sigmoid(20) ~ 1.0
frozen.input_weights[:] = 0; frozen.input_bias[:] = -20.0       # sigmoid(-20) ~ 0 -> write nothing
h, C = np.zeros(hidden_size), np.array([0.7, -0.4, 0.1, 0.9, -0.2])   # a memory we want to keep
original = C.copy()
for t in range(100):
    h, C, _ = frozen.step(inputs[t % sequence_length], h, C)
print("\nafter 100 steps with f~1, i~0 -> C unchanged:", np.allclose(C, original), np.round(C, 4))

hidden states h: (8, 5)  cell states C: (8, 5)

mean forget gate per step (near 1 -> memory kept): [0.784 0.731 0.736 0.69  0.713 0.733 0.573 0.785]

C grows by ACCUMULATION (unbounded), h is bounded by tanh * o:
  |C| max over steps: [0.213 0.226 0.272 0.258 0.296 0.296 0.515 0.525]
  |h| max over steps: [0.107 0.121 0.145 0.14  0.146 0.167 0.262 0.286]

after 100 steps with f~1, i~0 -> C unchanged: True [ 0.7 -0.4  0.1  0.9 -0.2]


In [20]:
import numpy as np

# Why the cell state fixes the gradient: compare the per-step multiplier over distance.
#   vanilla RNN :  dh_t/dh_{t-1} = diag(1 - h^2) W_hh   -> fixed by the WEIGHTS
#   LSTM        :  dC_t/dC_{t-1} = diag(f_t)            -> chosen by the NETWORK
def rnn_gradient_norm(distance: int, spectral_scale: float, hidden_size: int = 20, seed: int = 0) -> float:
    rng = np.random.default_rng(seed)
    recurrent = rng.standard_normal((hidden_size, hidden_size)) / np.sqrt(hidden_size)
    recurrent *= spectral_scale / max(abs(np.linalg.eigvals(recurrent)))
    hidden, jacobian = np.zeros(hidden_size), np.eye(hidden_size)
    for _ in range(distance):
        hidden = np.tanh(recurrent @ hidden + rng.standard_normal(hidden_size) * 0.5)
        jacobian = jacobian @ (np.diag(1 - hidden**2) @ recurrent)
    return np.linalg.norm(jacobian) / np.sqrt(hidden_size)

def lstm_gradient_norm(distance: int, forget_gate_value: float) -> float:
    return forget_gate_value ** distance          # prod of diag(f) -> just f^distance per slot

print("relative gradient magnitude reaching back `distance` steps\n")
print(f"{'distance':>9} | {'RNN sigma=0.9':>14} | {'LSTM f=0.5':>11} | {'LSTM f=0.95':>12} | {'LSTM f=0.999':>13}")
print("-" * 74)
for distance in [10, 25, 50, 100, 200]:
    print(f"{distance:>9} | {rnn_gradient_norm(distance, 0.9):>14.2e} | "
          f"{lstm_gradient_norm(distance, 0.5):>11.2e} | {lstm_gradient_norm(distance, 0.95):>12.2e} | "
          f"{lstm_gradient_norm(distance, 0.999):>13.2e}")

print("\nRNN            : dead within tens of steps, and nothing can be done about it.")
print("LSTM f=0.5     : vanishes just as fast -- gates are not magic (hence the positive b_f init).")
print("LSTM f->1      : gradient still ~intact at 200 steps. The network LEARNS to hold the gate open.")

relative gradient magnitude reaching back `distance` steps

 distance |  RNN sigma=0.9 |  LSTM f=0.5 |  LSTM f=0.95 |  LSTM f=0.999
--------------------------------------------------------------------------
       10 |       1.76e-02 |    9.77e-04 |     5.99e-01 |      9.90e-01
       25 |       5.59e-05 |    2.98e-08 |     2.77e-01 |      9.75e-01
       50 |       2.28e-09 |    8.88e-16 |     7.69e-02 |      9.51e-01
      100 |       1.24e-17 |    7.89e-31 |     5.92e-03 |      9.05e-01
      200 |       2.33e-35 |    6.22e-61 |     3.51e-05 |      8.19e-01

RNN            : dead within tens of steps, and nothing can be done about it.
LSTM f=0.5     : vanishes just as fast -- gates are not magic (hence the positive b_f init).
LSTM f->1      : gradient still ~intact at 200 steps. The network LEARNS to hold the gate open.


## Attention

### 1. Which problem are we solving?

The LSTM fixed the *gradient* problem but left the two **structural** ones untouched. Both come
from the same source: in a recurrent model, information from position `k` can only reach position
`t` by **travelling through every step in between**.

- **The bottleneck.** The entire prefix — or in seq2seq, the entire source sentence — is squeezed
  into one fixed-size vector. Translating a 40-word sentence through a single 512-dim `h` loses
  information no matter how good the gates are. Performance visibly collapsed on long sentences.
- **The sequential chain.** `h_t` needs `h_{t-1}`, so a length-`T` sequence costs `T` dependent
  steps that no GPU can parallelize. Distance between two positions is `t − k` operations.

**The purpose of attention, in one sentence:** *let every position read directly from every other
position, in a single step, with learned weights deciding how much of each to take.*

That single change dissolves all three problems at once:

- **No bottleneck** — nothing is compressed into one vector. All `T` representations stay
  available, and each position pulls what it needs from all of them.
- **Path length 1** — position `t` reaches position `k` in **one** operation, not `t − k`. So the
  gradient between any two positions is a single factor, not a product that decays with distance.
  (Compare the RNN's `∏ diag(1−h²)W_hh` and the LSTM's `∏ diag(f_i)` — attention has **no product
  at all**.)
- **Fully parallel** — every position's output is computed independently, so the whole sequence is
  one matrix multiply instead of `T` sequential steps.

**Historically** this arrived as a patch, not a replacement: Bahdanau et al. (2014) kept the RNN
encoder-decoder but let the decoder, at each output step, look back over **all** encoder states and
take a weighted average. The bottleneck vanished and long-sentence translation jumped. Three years
later, *Attention Is All You Need* made the obvious deletion — **remove the RNN entirely** and keep
only the attention. That's the Transformer, and it's what we're building.

### 2. Architecture and the math behind it

**The core idea: a soft dictionary lookup.** A Python dict does a *hard* lookup — the query either
matches a key exactly, or it doesn't:

```
d = {"cat": v1, "dog": v2}      d["cat"]  ->  v1     (exact match, all-or-nothing)
```

Attention does the same thing **softly and differentiably**: compare the query against **every**
key, turn the similarities into weights that sum to 1, and return the **weighted average of all
values**. A hard lookup is the limit case where one weight is 1 and the rest are 0.

**The three roles.** Every position produces three different vectors from its own representation:

- **Query `q`** — *what I'm looking for.* ("I'm a verb; where's my subject?")
- **Key `k`** — *what I offer, as an advertisement.* ("I'm a plural noun.")
- **Value `v`** — *what I actually hand over if selected.* The content, separate from the
  advertisement — so a token can be **easy to find** for one reason and **useful** for another.

$$Q = X W_Q, \qquad K = X W_K, \qquad V = X W_V$$

- `X` ∈ ℝ^(`T × d_model`) — the input: one row per position (token + positional embedding, Step 3).
- `W_Q, W_K` ∈ ℝ^(`d_model × d_k`), `W_V` ∈ ℝ^(`d_model × d_v`) — **the only learned parameters
  here**. Attention itself has no weights; the learning lives in these three projections.
- `Q, K, V` ∈ ℝ^(`T × d_k`) — every position gets all three roles simultaneously. In
  **self-attention** they all come from the *same* `X` (the sequence looks at itself); in
  **cross-attention** `Q` comes from one sequence and `K, V` from another.

**Step 1 — score every query against every key.**

$$S = \frac{Q K^\top}{\sqrt{d_k}}$$

- `Q Kᵀ` ∈ ℝ^(`T × T`) — `S[i,j]` = dot product of query `i` with key `j` = **how relevant
  position `j` is to position `i`**. The dot product is the similarity measure: large when the two
  vectors point the same way.
- This one matmul computes **all `T²` pairwise relevances at once** — the source of both the
  parallelism and the `O(T²)` cost.
- `√d_k` — **the scaling, and it is not cosmetic.** If the components of `q` and `k` are roughly
  independent with unit variance, their dot product over `d_k` dims has variance `d_k`, so typical
  magnitudes grow like `√d_k`. Feed scores of size ±30 into a softmax and it **saturates** into a
  near one-hot distribution — where its gradient is `p(1−p) ≈ 0`, so learning stalls. Dividing by
  `√d_k` restores unit variance and keeps the softmax in its responsive range. (See the second
  code cell.)

**Step 2 — turn scores into weights.**

$$A = \text{softmax}(S) \qquad \text{(row-wise)}$$

- `A` ∈ ℝ^(`T × T`), the **attention matrix**. Row `i` is a probability distribution: **each row
  sums to 1**, so every position distributes exactly one unit of attention across the sequence.
- `A[i,j]` = "how much position `i` reads from position `j`" — directly plottable as a heatmap,
  which is why Transformers are far more interpretable than LSTMs (Step 8).
- Softmax makes it **soft** (differentiable — you can backprop through the choice) and
  **competitive** (attending more to one position necessarily means less elsewhere).

**Step 3 — take the weighted average of the values.**

$$\text{Attention}(Q,K,V) = A V$$

- Output ∈ ℝ^(`T × d_v`) — same shape as the input: one vector per position, so blocks stack.
- Row `i` of the output is `Σ_j A[i,j] · V[j]` — a **convex combination of every value vector**,
  weighted by relevance. Nothing was compressed on the way in, and no `W_hh` stands between
  position `i` and position `j`.

Altogether, the formula you'll see everywhere:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

**What's missing is as important as what's there.** No recurrence, no convolution — and therefore
**no notion of order**: attention treats its input as a *set*, so shuffling the rows just shuffles
the outputs. That's precisely why positional embeddings (Step 3) are mandatory here and were
unnecessary for the RNN. Two consequences to carry into the next topics: nothing stops a position
from reading the **future**, which is what **causal masking** fixes; and one attention matrix can
only express one kind of relationship at a time, which is what **multi-head** fixes.

**The pipeline of calculations, in two scenarios.**

**(A) Learning (training).** The whole sequence is known upfront:

- Embed the tokens and add positional embeddings → `X`, shape `(T, d_model)`.
- Project once: `Q = XW_Q`, `K = XW_K`, `V = XW_V`. Three matmuls, all positions at once.
- One matmul `QKᵀ/√d_k` → the full `T × T` score matrix. **No loop over `t` anywhere.**
- Add the causal mask (`−∞` above the diagonal), softmax row-wise → `A`.
- One matmul `A V` → the output for every position simultaneously.
- Loss over all `T` positions, then backward through: `AV` matmul → softmax → `QKᵀ` → the three
  projections. Cache `Q, K, V, A` for it.
- **The headline:** every position is computed in parallel, in ~4 matmuls total, regardless of `T`.
  The RNN needed `T` dependent steps for the same work.

**(B) Actual working (generation).** One token at a time, since the next token isn't known yet:

- The new token produces **one** query row `q`, plus its own `k` and `v`.
- Append that `k, v` to the **KV-cache** — the keys and values of all previous positions, kept so
  they're never recomputed.
- Score `q` against **all cached keys**, softmax, weighted-average the cached values → one output
  vector → logits → sample the next token.
- Repeat. Without the cache you'd redo the entire prefix each step (`O(T²)` per token instead of
  `O(T)`).
- **The cost that isn't free:** the cache grows linearly with the sequence, so memory grows with
  length. The RNN carried one fixed-size `(h, C)` no matter how long the output — the one place
  recurrence still wins, and the reason context windows are finite.

### 3. Applications (when to use)

Attention is the default mechanism for relating elements of a set or sequence:

- **Language models** — GPT and friends are stacks of self-attention (Steps 4–5, our model)
- **Machine translation** — where it was invented; cross-attention lets the decoder read the source
- **Vision (ViT)** — patches as tokens, attention over them (Step 6)
- **Multimodal / captioning — our task:** attention over a joint sequence of image and text tokens
  lets a caption word read directly from the relevant patch. *Show, Attend and Tell* (2015) did
  this over CNN features; the visualizations of a word attending to its object are the
  **grounding** analysis in Step 8.
- **Speech, protein folding (AlphaFold), graphs, recommenders** — anywhere pairwise relationships
  matter more than fixed local structure

**Use it when:** relationships are long-range or content-dependent, and sequences are short enough
that `T²` is affordable. **Prefer recurrence when:** the stream is unbounded or truly online, or
memory per token must stay constant.

### 4. Advantages

- **Constant path length between any two positions** — one operation, so no distance-based decay.
  This is the property RNNs and LSTMs could never buy.
- **Fully parallel across positions** — the whole sequence in a few matmuls. This, more than
  accuracy, is why Transformers took over: they use hardware that recurrence cannot.
- **No information bottleneck** — all `T` representations stay available; nothing is squeezed into
  a single summary vector.
- **Content-based, dynamic routing** — which positions matter is computed *from the data* at every
  step, not fixed by architecture like a convolution's window.
- **Interpretable** — `A` is a `T × T` matrix you can plot and read.
- **Flexible over modalities** — anything you can turn into a set of vectors can attend to anything
  else, which is exactly what makes image + text in one sequence possible.

### 5. Disadvantages

- **`O(T²)` time and memory.** The score matrix is `T × T`; doubling the sequence quadruples the
  cost. This single fact sets the context-window limit of every LLM and motivates FlashAttention,
  sparse/linear attention, and the rest.
- **No built-in sense of order** — it's a set operation, so positional information must be injected
  by hand (Step 3). Get this wrong and word order stops mattering entirely.
- **Growing KV-cache at generation** — memory scales with sequence length, unlike the RNN's `O(1)`.
- **Data-hungry** — very few inductive biases (no locality, no recency, no order). It has to learn
  from data what a CNN or RNN gets for free, so it needs more of it.
- **One attention matrix expresses one relationship** — a single head must average over syntax,
  coreference, position, and topic at once. (Fixed by multi-head, next topic.)
- **The hardest backward pass in the project** — gradients flow back through the `AV` matmul, the
  row-wise softmax Jacobian, *and* `QKᵀ` into three separate projections.

> **Next:** two limitations above are load-bearing enough to be their own topics — **causal
> masking** (stop position `t` from reading the future, which is what makes parallel training of a
> language model legal) and **multi-head attention** (run several attention operations in parallel
> so different heads can specialize).

### 6. Useful resources

- Vaswani et al. (2017) — **Attention Is All You Need** — *the original paper*
  <https://arxiv.org/abs/1706.03762> — §3.2 is scaled dot-product and multi-head attention;
  short and readable, and footnote 4 is where the `√d_k` argument comes from.
- Alammar — **The Illustrated Transformer**
  <https://jalammar.github.io/illustrated-transformer/> — the diagrams everyone learns Q/K/V
  from; read alongside the paper.
- Karpathy — **Let's build GPT: from scratch, in code, spelled out**
  <https://www.youtube.com/watch?v=kCc8FmEb1nY> — builds exactly this, live, with the causal mask
  and multi-head that come next.
- **Where attention was actually introduced:** Bahdanau, Cho & Bengio (2014) — **Neural Machine
  Translation by Jointly Learning to Align and Translate** <https://arxiv.org/abs/1409.0473> —
  attention as the fix for the seq2seq bottleneck of §1, three years before the Transformer.
- Olah & Carter — **Attention and Augmented Recurrent Neural Networks**
  <https://distill.pub/2016/augmented-rnns/> — attention as differentiable memory access,
  interactive.

In [21]:
import numpy as np

def softmax_rows(scores: np.ndarray) -> np.ndarray:
    # Row-wise stable softmax (Step 0's log-sum-exp trick), one distribution per query.
    shifted = scores - scores.max(axis=-1, keepdims=True)
    exponentiated = np.exp(shifted)
    return exponentiated / exponentiated.sum(axis=-1, keepdims=True)

def scaled_dot_product_attention(queries: np.ndarray, keys: np.ndarray, values: np.ndarray) -> tuple:
    key_dimension: int = queries.shape[-1]
    scores = queries @ keys.T / np.sqrt(key_dimension)   # (T, T)  every query vs every key
    attention_weights = softmax_rows(scores)              # (T, T)  each ROW sums to 1
    output = attention_weights @ values                  # (T, d)  weighted average of ALL values
    return output, attention_weights

# --- a hand-built example so the numbers mean something -------------------------------
# 4 positions. Keys advertise a "topic"; the query of position 3 looks for topic A.
#            topic_a  topic_b
keys = np.array([[1.0, 0.0],     # pos 0: about A
                 [0.0, 1.0],     # pos 1: about B
                 [1.0, 0.0],     # pos 2: about A
                 [0.0, 1.0]])    # pos 3: about B
values = np.array([[10.0, 0.0],  # the CONTENT each position hands over if selected
                   [0.0, 10.0],
                   [20.0, 0.0],
                   [0.0, 20.0]])
queries = np.array([[3.0, 0.0],  # pos 0 asks for A
                    [0.0, 3.0],  # pos 1 asks for B
                    [3.0, 0.0],  # pos 2 asks for A
                    [3.0, 0.0]]) # pos 3 asks for A  <- reads from positions 0 and 2

output, attention_weights = scaled_dot_product_attention(queries, keys, values)

print("attention matrix A (row i = how much position i reads from each position):")
print(np.round(attention_weights, 3))
print("\nevery row sums to 1:", np.allclose(attention_weights.sum(axis=1), 1))
print("\nrow 3 attends to positions 0 and 2 (the topic-A ones):", np.round(attention_weights[3], 3))
print("output[3] = weighted average of the values:", np.round(output[3], 2))
print("manual check                                :",
      np.round(sum(attention_weights[3, j] * values[j] for j in range(4)), 2))

# --- attention is a SET operation: no notion of order --------------------------------
permutation = [2, 0, 3, 1]
out_permuted, _ = scaled_dot_product_attention(queries[permutation], keys[permutation], values[permutation])
print("\nshuffling the positions just shuffles the outputs:",
      np.allclose(out_permuted, output[permutation]),
      " <- why positional embeddings are mandatory")

# --- path length 1: position i reaches position j directly ---------------------------
print("nothing between position 3 and position 0 -- one matmul, no product over distance.")

attention matrix A (row i = how much position i reads from each position):
[[0.446 0.054 0.446 0.054]
 [0.054 0.446 0.054 0.446]
 [0.446 0.054 0.446 0.054]
 [0.446 0.054 0.446 0.054]]

every row sums to 1: True

row 3 attends to positions 0 and 2 (the topic-A ones): [0.446 0.054 0.446 0.054]
output[3] = weighted average of the values: [13.39  1.61]
manual check                                : [13.39  1.61]

shuffling the positions just shuffles the outputs: True  <- why positional embeddings are mandatory
nothing between position 3 and position 0 -- one matmul, no product over distance.


In [22]:
import numpy as np

# Why divide by sqrt(d_k)?  Dot products grow like sqrt(d_k), and a softmax fed large
# numbers saturates into a near one-hot -- where its gradient p*(1-p) is ~0 and learning stalls.
rng = np.random.default_rng(0)

def softmax_row(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - x.max()); return e / e.sum()

print(f"{'d_k':>5} | {'std(q.k)':>8} {'sqrt(d_k)':>9} | {'max weight':>21} | {'softmax gradient scale':>24}")
print(f"{'':>5} | {'measured':>8} {'predicted':>9} | {'unscaled':>9} {'scaled':>11} | {'unscaled':>11} {'scaled':>12}")
print("-" * 84)

for key_dimension in [4, 16, 64, 256, 1024]:
    # measure the spread of the dot product over MANY random pairs -- it tracks sqrt(d_k)
    sample_queries = rng.standard_normal((2000, key_dimension))
    sample_keys    = rng.standard_normal((2000, key_dimension))
    measured_std = (sample_queries * sample_keys).sum(axis=1).std()

    query = rng.standard_normal(key_dimension)                     # unit-variance components
    keys  = rng.standard_normal((8, key_dimension))                # 8 candidate positions
    raw_scores = keys @ query                                      # dot products
    scaled_scores = raw_scores / np.sqrt(key_dimension)              # the fix

    unscaled_weights = softmax_row(raw_scores)
    scaled_weights   = softmax_row(scaled_scores)
    # softmax's own gradient magnitude: p*(1-p) -- collapses to 0 when one weight dominates
    print(f"{key_dimension:>5} | {measured_std:>8.1f} {np.sqrt(key_dimension):>9.1f} | "
          f"{unscaled_weights.max():>9.3f} {scaled_weights.max():>11.3f} | "
          f"{(unscaled_weights*(1-unscaled_weights)).max():>11.2e} "
          f"{(scaled_weights*(1-scaled_weights)).max():>12.2e}")

print("\nmeasured std matches sqrt(d_k) exactly -- that is why sqrt(d_k) is the divisor.")
print("Unscaled at large d_k: one weight -> 1.0, gradient -> 0. The layer stops learning.")
print("Scaled: the distribution stays responsive at every d_k.")

  d_k | std(q.k) sqrt(d_k) |            max weight |   softmax gradient scale
      | measured predicted |  unscaled      scaled |    unscaled       scaled
------------------------------------------------------------------------------------
    4 |      2.0       2.0 |     0.435       0.266 |    2.46e-01     1.95e-01
   16 |      4.0       4.0 |     0.701       0.277 |    2.10e-01     2.00e-01
   64 |      8.0       8.0 |     0.999       0.363 |    1.47e-03     2.31e-01
  256 |     15.8      16.0 |     0.993       0.258 |    6.68e-03     1.91e-01
 1024 |     31.9      32.0 |     1.000       0.387 |    2.22e-15     2.37e-01

measured std matches sqrt(d_k) exactly -- that is why sqrt(d_k) is the divisor.
Unscaled at large d_k: one weight -> 1.0, gradient -> 0. The layer stops learning.
Scaled: the distribution stays responsive at every d_k.


## Step 4 — topics not yet written up

Three attention topics are **deliberately skipped here** and left to be learned while implementing
(LEARNING.md Step 4 practice list). Noted so the gap is explicit, not forgotten:

- **Causal masking** — set scores above the diagonal to `−∞` before the softmax so position `t`
  cannot read the future. What makes it legal to train on all `T` positions in parallel.
- **Multi-head attention** — run `h` attentions in parallel on `d_k = d/h` slices, concatenate,
  project. Fixes the "one matrix expresses one relationship" limitation in the attention §5.
- **Backprop through attention** — back through `AV`, the row-wise softmax Jacobian, and `QKᵀ`
  into the three projections. The hardest backward in the project.

*(Theory to be filled in later if needed — the plan is to learn these by writing the code.)*

# Step 5 — Normalization, Blocks & Deep Nets

Attention is the mechanism; this step is the **packaging** that makes it trainable at depth.
LayerNorm, residual connections and the MLP sub-layer are what turn one attention operation into
a block you can stack N times — which is the Transformer.

## Layer Normalization

### 1. Which problem are we solving?

We now have every piece of one attention operation. The moment we stack them, a new problem
appears that has nothing to do with attention itself: **activation scale drifts**.

Each layer's output is the next layer's input. Nothing constrains its magnitude, so scales
compound through depth — one layer's weights being slightly too large multiplies through every
layer after it. Concretely this breaks the pieces we just built:

- **Softmax saturates.** Attention scores are dot products of activations; if activations grow,
  scores grow, and `softmax` collapses to near one-hot where its gradient is `p(1−p) ≈ 0`. We
  scaled by `√d_k` to fix this *at initialization* — but nothing keeps it fixed during training.
- **Gradients explode or vanish** through the depth, the Step 1 problem returning at scale.
- **Every layer chases a moving target.** As earlier layers update, the *distribution* of inputs
  to later layers shifts, so they must constantly re-adapt (the "internal covariate shift"
  argument from the BatchNorm paper).

The fix is to **re-standardize activations at fixed points in the network** — force a known mean
and variance, then let the model learn to scale away from that if it wants.

**Why not BatchNorm?** It normalizes each feature across the **batch**, which is wrong for us:

- It makes one example's output depend on the **other examples in the batch** — sequences of
  different lengths and padding poison the statistics.
- It needs **running averages** for inference, so train and test behave differently.
- It degrades badly at small batch sizes, and is awkward for variable-length sequences.

**LayerNorm's answer:** normalize across the **feature dimension of a single token**, using only
that token's own `d` numbers. No batch, no other positions, no running statistics.

### 2. Architecture and the math behind it

LayerNorm operates on **one vector at a time** — one token's `d`-dimensional representation —
and does the same thing to every token independently.

**Step 1 — the statistics, computed over that token's own features:**

$$\mu = \frac{1}{d}\sum_{i=1}^{d} x_i, \qquad \sigma^2 = \frac{1}{d}\sum_{i=1}^{d} (x_i - \mu)^2$$

- `x` ∈ ℝ^`d` — one token's vector. `d` is the **feature/model dimension**, not the batch size.
- `μ`, `σ²` — **scalars**, one pair per token. Computed *along* the features, so every token in
  every sequence in the batch gets its own.
- **This is the whole difference from BatchNorm**, which computes one `μ, σ²` per *feature* across
  the batch. Same formula, different axis — and the axis is the entire design decision.

**Step 2 — normalize:**

$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$$

- Subtracting `μ` centres the vector; dividing by the standard deviation sets its scale.
- Result: `x̂` has **mean 0 and variance 1 across its `d` features**, guaranteed, for every token.
- `ε` (≈`1e-5`) — prevents division by zero when a token's features are all equal (`σ² = 0`), and
  keeps the gradient finite there.

**Step 3 — scale and shift, learnably:**

$$y_i = \gamma_i \,\hat{x}_i + \beta_i$$

- `γ, β` ∈ ℝ^`d` — **the only parameters**, one per feature, shared across all tokens and
  sequences. Initialized `γ = 1`, `β = 0`, so LayerNorm starts as pure normalization.
- **Why undo the normalization we just did?** Because mean 0 / variance 1 is not always the best
  input distribution — for a sigmoid it means sitting in the linear region. `γ, β` let the network
  *choose* the scale rather than having it imposed, while keeping it **decoupled from the
  incoming magnitude**. Normalization removes the drift; `γ, β` restore the expressiveness.

**The backward pass** is the tricky part, and worth understanding rather than memorizing. The
reason it isn't elementwise: `μ` and `σ²` depend on **every** `x_i`, so changing one input changes
*all* `d` outputs. With `dx̂ = dy ⊙ γ`:

$$\frac{\partial L}{\partial x} = \frac{1}{d\,\sqrt{\sigma^2+\epsilon}}\left[\, d\,\widehat{dx} \;-\; \sum_{j} \widehat{dx}_j \;-\; \hat{x}\sum_{j} \widehat{dx}_j\,\hat{x}_j \,\right]$$

- **First term** — the direct, elementwise path.
- **Second term** — `Σ dx̂`, the correction from `μ`: raising one input raises the mean, which
  lowers every output. Subtracting the sum removes the component of the gradient that would just
  shift the whole vector.
- **Third term** — `x̂ Σ (dx̂ ⊙ x̂)`, the correction from `σ²`: raising one input changes the
  variance, rescaling every output. This removes the component that would just rescale the vector.
- Read together: **LayerNorm's backward projects out the gradient directions that only change the
  mean or the scale** — precisely the two things it normalized away. Gradients that would merely
  shift or stretch the vector don't get through.
- The parameter gradients are simple sums over every token the layer saw:
  `dγ = Σ_tokens dy ⊙ x̂` and `dβ = Σ_tokens dy`.

**The pipeline of calculations, in two scenarios.** This is the short section, and that's the
point:

**(A) Learning (training).**

- For each token vector independently: compute `μ` and `σ²` over its `d` features, normalize,
  scale by `γ`, shift by `β`. All tokens in the batch in parallel — it's a per-row operation.
- Cache `x̂` and `1/√(σ²+ε)` for the backward pass.
- Backward: the three-term formula above for `dx`; accumulate `dγ`, `dβ` by summing over every
  token position and every sequence in the batch.
- `γ` and `β` are updated by Adam like any other parameter.

**(B) Actual working (inference).**

- **Exactly the same computation.** Statistics come from the current token's own features, which
  are available at inference just as they were at training.
- **No running averages, no train/eval mode, no batch dependence** — a single token normalizes
  identically whether it arrives alone or in a batch of 512.
- This is a genuine advantage over BatchNorm, whose train and inference paths differ and which is
  a classic source of "works in training, breaks in production" bugs.

**Variant worth knowing — RMSNorm.** Drops the mean subtraction entirely and divides by the root
mean square: `y = γ ⊙ x / √(mean(x²) + ε)`. Cheaper, no `β`, and works about as well — LLaMA and
most recent LLMs use it, which suggests the **re-centring was never the important part; the
re-scaling was**.

### 3. Applications (when to use)

- **Every Transformer** — two LayerNorms per block (before attention and before the MLP, in the
  pre-norm arrangement we'll use). It is the standard normalization for sequence models.
- **RNNs/LSTMs** — its original target: BatchNorm has no good answer for recurrence, LayerNorm
  applies unchanged at every time step.
- **Small or variable batch sizes** — reinforcement learning, online learning, batch size 1.
- **Not the default for CNNs** — vision with large batches still often prefers BatchNorm; the
  spatial equivalents are GroupNorm and InstanceNorm.

**Rule of thumb:** normalize over the **batch** when batches are large and i.i.d.; normalize over
**features** when examples are sequences, variable length, or few.

### 4. Advantages

- **Identical at train and inference** — no running statistics, no mode switch, no train/test skew.
- **Independent of batch size and composition** — works at batch size 1; one example's result never
  depends on its neighbours.
- **Handles variable-length sequences and padding naturally** — statistics are per token.
- **Stabilizes and accelerates training** — keeps activations in a fixed range so the softmax stays
  responsive and gradients stay well-scaled; permits higher learning rates.
- **Cheap** — two passes over `d` numbers per token, `2d` parameters per layer.
- **Preserves expressiveness** via `γ, β` — normalization is a re-parameterization, not a
  restriction.

### 5. Disadvantages

- **The backward is genuinely non-trivial** — the mean/variance coupling makes every output depend
  on every input, so a naive elementwise backward is silently wrong. (This is why LEARNING.md sends
  you to gradient-check it.)
- **Not free at inference** — unlike BatchNorm, it *cannot* be folded into the preceding linear
  layer, because its statistics are data-dependent. In deployed LLMs, normalization is a
  measurable slice of runtime.
- **The mean subtraction may be wasted work** — RMSNorm drops it with no real loss.
- **Placement matters more than the layer itself** — pre-norm vs post-norm changes whether a deep
  Transformer trains at all without warmup (the next topic).
- **Can mask underlying problems** — it will paper over bad initialization or a bad learning rate
  rather than letting them fail loudly.
- **Assumes features are comparable** — normalizing across a token's dimensions is only sensible
  because they're a learned, homogeneous representation; you would not do this to raw
  heterogeneous features (age, income, pixel count).

### 6. Useful resources

- **Original paper:** Ba, Kiros & Hinton (2016) — **Layer Normalization**
  <https://arxiv.org/abs/1607.06450> — introduces it explicitly as the fix for BatchNorm's
  batch-dependence in RNNs; §3 is the comparison that motivates the axis choice.
- **BatchNorm — the paper it reacts to:** Ioffe & Szegedy (2015) — **Batch Normalization**
  <https://arxiv.org/abs/1502.03167> — where "internal covariate shift" comes from.
- **RMSNorm — original paper:** Zhang & Sennrich (2019) — **Root Mean Square Layer Normalization**
  <https://arxiv.org/abs/1910.07467> — the "re-centring wasn't needed" result.
- Xiong et al. (2020) — **On Layer Normalization in the Transformer Architecture**
  <https://arxiv.org/abs/2002.04745> — proves why **pre-norm** trains without warmup and post-norm
  doesn't. Read before the next topic.
- Xu et al. (2019) — **Understanding and Improving Layer Normalization**
  <https://arxiv.org/abs/1911.07013> — evidence that the **backward** (the gradient projection
  above), not the forward normalization, is what actually helps.

In [23]:
import numpy as np

class LayerNorm:
    """Normalize each token vector over its own `d` features, then scale and shift.

    Shapes: x (..., d) -> y (..., d).  Parameters: gamma (d,), beta (d,).
    """

    def __init__(self, feature_dimension: int, epsilon: float = 1e-5) -> None:
        self.epsilon: float = epsilon
        self.gamma: np.ndarray = np.ones(feature_dimension)    # start as pure normalization
        self.beta: np.ndarray = np.zeros(feature_dimension)
        self.normalized: np.ndarray | None = None              # cached x_hat, for backward
        self.inverse_std: np.ndarray | None = None             # cached 1/sqrt(var + eps)
        self.d_gamma: np.ndarray | None = None
        self.d_beta: np.ndarray | None = None

    def forward(self, x: np.ndarray) -> np.ndarray:
        mean = x.mean(axis=-1, keepdims=True)                  # per TOKEN, over features
        variance = x.var(axis=-1, keepdims=True)
        self.inverse_std = 1.0 / np.sqrt(variance + self.epsilon)
        self.normalized = (x - mean) * self.inverse_std        # mean 0, variance 1 per row
        return self.gamma * self.normalized + self.beta

    def backward(self, d_y: np.ndarray) -> np.ndarray:
        feature_dimension = d_y.shape[-1]
        # parameters: sum over every token this layer saw
        self.d_gamma = (d_y * self.normalized).reshape(-1, feature_dimension).sum(axis=0)
        self.d_beta = d_y.reshape(-1, feature_dimension).sum(axis=0)

        d_normalized = d_y * self.gamma
        # the three terms: direct path, mean correction, variance correction
        return (self.inverse_std / feature_dimension) * (
            feature_dimension * d_normalized
            - d_normalized.sum(axis=-1, keepdims=True)
            - self.normalized * (d_normalized * self.normalized).sum(axis=-1, keepdims=True)
        )

rng = np.random.default_rng(0)
x = rng.standard_normal((4, 6)) * 10 + 3        # 4 tokens, d=6, deliberately off-scale
layer_norm = LayerNorm(feature_dimension=6)
y = layer_norm.forward(x)

print("input  mean/std per token:", np.round(x.mean(-1), 2), np.round(x.std(-1), 2))
print("output mean/std per token:", np.round(y.mean(-1), 6), np.round(y.std(-1), 4))
print("-> every token forced to mean 0, variance 1, regardless of what came in")

# Independence: a token's output does NOT depend on the other tokens in the batch.
single_token = layer_norm.forward(x[1:2])
print("\nsame token alone vs in a batch of 4:", np.allclose(single_token, y[1:2]),
      " <- impossible with BatchNorm")

# Nor on the batch ORDER
shuffled = layer_norm.forward(x[[2, 0, 3, 1]])
print("batch order is irrelevant           :", np.allclose(shuffled, y[[2, 0, 3, 1]]))

# gamma / beta let the model choose a different scale
layer_norm.gamma = np.full(6, 2.0); layer_norm.beta = np.full(6, 5.0)
rescaled = layer_norm.forward(x)
print("\nwith gamma=2, beta=5 -> mean/std per token:",
      np.round(rescaled.mean(-1), 3), np.round(rescaled.std(-1), 3))

input  mean/std per token: [ 3.94  2.5  -5.97  6.96] [3.69 9.19 7.2  6.79]
output mean/std per token: [ 0.  0. -0. -0.] [1. 1. 1. 1.]
-> every token forced to mean 0, variance 1, regardless of what came in

same token alone vs in a batch of 4: True  <- impossible with BatchNorm
batch order is irrelevant           : True

with gamma=2, beta=5 -> mean/std per token: [5. 5. 5. 5.] [2. 2. 2. 2.]


In [24]:
import numpy as np

# Gradient-check the tricky backward: the mean/variance coupling makes every output depend on
# every input, so an elementwise backward is silently WRONG. Finite differences settle it.
def numeric_gradient(f, x: np.ndarray, h: float = 1e-6) -> np.ndarray:
    gradient = np.zeros_like(x, dtype=float)
    flat = x.reshape(-1)
    for i in range(flat.size):
        original = flat[i]
        flat[i] = original + h; plus = f(x)
        flat[i] = original - h; minus = f(x)
        flat[i] = original
        gradient.reshape(-1)[i] = (plus - minus) / (2 * h)
    return gradient

rng = np.random.default_rng(1)
x = rng.standard_normal((3, 5))
upstream = rng.standard_normal((3, 5))          # random upstream gradient, not all ones

layer_norm = LayerNorm(feature_dimension=5)
layer_norm.gamma = rng.standard_normal(5)       # non-trivial gamma/beta
layer_norm.beta = rng.standard_normal(5)

layer_norm.forward(x)
analytic_dx = layer_norm.backward(upstream)

# scalar loss L = sum(y * upstream) so that dL/dy = upstream
def loss_of_x(x_current: np.ndarray) -> float:
    return float(np.sum(layer_norm.forward(x_current) * upstream))

numeric_dx = numeric_gradient(loss_of_x, x.copy())
print("dx  max abs difference:", np.abs(analytic_dx - numeric_dx).max())

layer_norm.forward(x); layer_norm.backward(upstream)
def loss_of_gamma(gamma_current: np.ndarray) -> float:
    saved = layer_norm.gamma; layer_norm.gamma = gamma_current
    value = float(np.sum(layer_norm.forward(x) * upstream)); layer_norm.gamma = saved
    return value
print("dgamma max abs difference:", np.abs(layer_norm.d_gamma - numeric_gradient(loss_of_gamma, layer_norm.gamma.copy())).max())

# What a NAIVE (wrong) backward would give -- ignoring the mean/variance coupling
naive_dx = upstream * layer_norm.gamma * layer_norm.inverse_std
print("\nnaive elementwise backward, max abs error:", np.abs(naive_dx - numeric_dx).max().round(4))
print("-> visibly wrong. The two correction terms are not optional.")

# Why those correction terms exist, seen from the forward side: LayerNorm is INVARIANT to
# shifting or rescaling a token, so gradients in those directions are useless and get projected out.
constant_shift = np.ones_like(x) * 3.7
baseline = layer_norm.forward(x).copy()
shift_difference = np.abs(layer_norm.forward(x + constant_shift) - baseline).max()
scale_difference = np.abs(layer_norm.forward(x * 10) - baseline).max()
print(f"\nshift a token by +3.7 -> output changes by {shift_difference:.2e}  (exactly invariant)")
print(f"scale a token by 10x  -> output changes by {scale_difference:.2e}  (invariant up to eps)")
print("-> those two invariances are exactly what the mean and variance terms in backward encode:")
print("   gradient components that would only shift or rescale a token are projected away.")

dx  max abs difference: 5.608941089363384e-10
dgamma max abs difference: 4.1068703993119016e-10

naive elementwise backward, max abs error: 2.6486
-> visibly wrong. The two correction terms are not optional.

shift a token by +3.7 -> output changes by 2.66e-15  (exactly invariant)
scale a token by 10x  -> output changes by 6.84e-05  (invariant up to eps)
-> those two invariances are exactly what the mean and variance terms in backward encode:
   gradient components that would only shift or rescale a token are projected away.
